In [ ]:
# Python Notebook - Car_Sales_Analysis

"""
# New Notebook
"""

datasets

df = datasets["uk_used_car_listings_staging"]

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

import sqlite3
import pandas as pd

# Create a temporary SQL database
conn = sqlite3.connect(":memory:")

# Copy the Mode dataset into a SQL table
df.to_sql(
    "used_cars",
    conn,
    index=False,
    if_exists="replace"
)

# Confirm that all records reached SQL
sql_table_check = pd.read_sql_query("""
    SELECT
        COUNT(*) AS total_rows
    FROM used_cars
""", conn)

sql_table_check

missing_values_sql = """
SELECT
    'manufacturer' AS field_name,
    SUM(CASE
        WHEN manufacturer IS NULL OR TRIM(manufacturer) = ''
        THEN 1 ELSE 0
    END) AS missing_count
FROM used_cars

UNION ALL

SELECT
    'model',
    SUM(CASE
        WHEN model IS NULL OR TRIM(model) = ''
        THEN 1 ELSE 0
    END)
FROM used_cars

UNION ALL

SELECT
    'year',
    SUM(CASE WHEN year IS NULL THEN 1 ELSE 0 END)
FROM used_cars

UNION ALL

SELECT
    'price',
    SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END)
FROM used_cars

UNION ALL

SELECT
    'transmission',
    SUM(CASE
        WHEN transmission IS NULL OR TRIM(transmission) = ''
        THEN 1 ELSE 0
    END)
FROM used_cars

UNION ALL

SELECT
    'mileage',
    SUM(CASE WHEN mileage IS NULL THEN 1 ELSE 0 END)
FROM used_cars

UNION ALL

SELECT
    'fuel_type',
    SUM(CASE
        WHEN fuel_type IS NULL OR TRIM(fuel_type) = ''
        THEN 1 ELSE 0
    END)
FROM used_cars

UNION ALL

SELECT
    'tax',
    SUM(CASE WHEN tax IS NULL THEN 1 ELSE 0 END)
FROM used_cars

UNION ALL

SELECT
    'mpg',
    SUM(CASE WHEN mpg IS NULL THEN 1 ELSE 0 END)
FROM used_cars

UNION ALL

SELECT
    'engine_size',
    SUM(CASE WHEN engine_size IS NULL THEN 1 ELSE 0 END)
FROM used_cars

UNION ALL

SELECT
    'source_file',
    SUM(CASE
        WHEN source_file IS NULL OR TRIM(source_file) = ''
        THEN 1 ELSE 0
    END)
FROM used_cars;
"""

missing_values = pd.read_sql_query(
    missing_values_sql,
    conn
)

missing_values["missing_percentage"] = (
    missing_values["missing_count"]
    / len(df)
    * 100
).round(2)

missing_values

duplicate_summary_sql = """
WITH repeated_groups AS (
    SELECT
        manufacturer,
        model,
        year,
        price,
        transmission,
        mileage,
        fuel_type,
        tax,
        mpg,
        engine_size,
        COUNT(*) AS record_count
    FROM used_cars
    GROUP BY
        manufacturer,
        model,
        year,
        price,
        transmission,
        mileage,
        fuel_type,
        tax,
        mpg,
        engine_size
    HAVING COUNT(*) > 1
)

SELECT
    COUNT(*) AS identical_record_groups,
    SUM(record_count) AS rows_in_identical_groups,
    SUM(record_count - 1) AS excess_identical_rows,
    ROUND(
        100.0 * SUM(record_count - 1)
        / (SELECT COUNT(*) FROM used_cars),
        2
    ) AS excess_rows_percentage,
    MAX(record_count) AS largest_identical_group
FROM repeated_groups;
"""

duplicate_summary = pd.read_sql_query(
    duplicate_summary_sql,
    conn
)

duplicate_summary

largest_duplicate_groups_sql = """
SELECT
    manufacturer,
    model,
    year,
    price,
    transmission,
    mileage,
    fuel_type,
    tax,
    mpg,
    engine_size,
    COUNT(*) AS record_count
FROM used_cars
GROUP BY
    manufacturer,
    model,
    year,
    price,
    transmission,
    mileage,
    fuel_type,
    tax,
    mpg,
    engine_size
HAVING COUNT(*) > 1
ORDER BY
    record_count DESC,
    manufacturer,
    model
LIMIT 20;
"""

largest_duplicate_groups = pd.read_sql_query(
    largest_duplicate_groups_sql,
    conn
)

largest_duplicate_groups

duplicates_by_manufacturer_sql = """
WITH manufacturer_totals AS (
    SELECT
        manufacturer,
        COUNT(*) AS manufacturer_rows
    FROM used_cars
    GROUP BY manufacturer
),

repeated_groups AS (
    SELECT
        manufacturer,
        model,
        year,
        price,
        transmission,
        mileage,
        fuel_type,
        tax,
        mpg,
        engine_size,
        COUNT(*) AS record_count
    FROM used_cars
    GROUP BY
        manufacturer,
        model,
        year,
        price,
        transmission,
        mileage,
        fuel_type,
        tax,
        mpg,
        engine_size
    HAVING COUNT(*) > 1
)

SELECT
    r.manufacturer,
    t.manufacturer_rows,
    COUNT(*) AS identical_groups,
    SUM(r.record_count) AS rows_in_identical_groups,
    SUM(r.record_count - 1) AS excess_identical_rows,

    ROUND(
        100.0 * SUM(r.record_count - 1)
        / t.manufacturer_rows,
        2
    ) AS excess_percentage_within_manufacturer,

    SUM(
        CASE
            WHEN r.mileage <= 1000
            THEN r.record_count - 1
            ELSE 0
        END
    ) AS excess_rows_at_1000_miles_or_less

FROM repeated_groups AS r

JOIN manufacturer_totals AS t
    ON r.manufacturer = t.manufacturer

GROUP BY
    r.manufacturer,
    t.manufacturer_rows

ORDER BY
    excess_identical_rows DESC;
"""

duplicates_by_manufacturer = pd.read_sql_query(
    duplicates_by_manufacturer_sql,
    conn
)

duplicates_by_manufacturer

create_deduplicated_table_sql = """
DROP TABLE IF EXISTS used_cars_deduplicated;

CREATE TABLE used_cars_deduplicated AS

SELECT
    manufacturer,
    model,
    year,
    price,
    transmission,
    mileage,
    fuel_type,
    tax,
    mpg,
    engine_size,
    MIN(source_file) AS source_file,
    COUNT(*) AS exact_record_count

FROM used_cars

GROUP BY
    manufacturer,
    model,
    year,
    price,
    transmission,
    mileage,
    fuel_type,
    tax,
    mpg,
    engine_size;
"""

conn.executescript(create_deduplicated_table_sql)

deduplication_check_sql = """
SELECT
    COUNT(*) AS deduplicated_rows,
    SUM(exact_record_count) AS original_rows_represented,
    SUM(CASE WHEN exact_record_count > 1 THEN 1 ELSE 0 END)
        AS repeated_record_groups,
    SUM(exact_record_count - 1) AS collapsed_excess_rows,
    MAX(exact_record_count) AS largest_identical_group
FROM used_cars_deduplicated;
"""

deduplication_check = pd.read_sql_query(
    deduplication_check_sql,
    conn
)

deduplication_check

numeric_range_audit_sql = """
SELECT
    'year' AS field_name,
    COUNT(*) AS record_count,
    COUNT(DISTINCT year) AS distinct_values,
    MIN(year) AS minimum_value,
    ROUND(AVG(year), 2) AS average_value,
    MAX(year) AS maximum_value,
    SUM(CASE WHEN year = 0 THEN 1 ELSE 0 END) AS zero_records,
    SUM(CASE WHEN year < 0 THEN 1 ELSE 0 END) AS negative_records
FROM used_cars_deduplicated

UNION ALL

SELECT
    'price',
    COUNT(*),
    COUNT(DISTINCT price),
    MIN(price),
    ROUND(AVG(price), 2),
    MAX(price),
    SUM(CASE WHEN price = 0 THEN 1 ELSE 0 END),
    SUM(CASE WHEN price < 0 THEN 1 ELSE 0 END)
FROM used_cars_deduplicated

UNION ALL

SELECT
    'mileage',
    COUNT(*),
    COUNT(DISTINCT mileage),
    MIN(mileage),
    ROUND(AVG(mileage), 2),
    MAX(mileage),
    SUM(CASE WHEN mileage = 0 THEN 1 ELSE 0 END),
    SUM(CASE WHEN mileage < 0 THEN 1 ELSE 0 END)
FROM used_cars_deduplicated

UNION ALL

SELECT
    'tax',
    COUNT(*),
    COUNT(DISTINCT tax),
    MIN(tax),
    ROUND(AVG(tax), 2),
    MAX(tax),
    SUM(CASE WHEN tax = 0 THEN 1 ELSE 0 END),
    SUM(CASE WHEN tax < 0 THEN 1 ELSE 0 END)
FROM used_cars_deduplicated

UNION ALL

SELECT
    'mpg',
    COUNT(*),
    COUNT(DISTINCT mpg),
    MIN(mpg),
    ROUND(AVG(mpg), 2),
    MAX(mpg),
    SUM(CASE WHEN mpg = 0 THEN 1 ELSE 0 END),
    SUM(CASE WHEN mpg < 0 THEN 1 ELSE 0 END)
FROM used_cars_deduplicated

UNION ALL

SELECT
    'engine_size',
    COUNT(*),
    COUNT(DISTINCT engine_size),
    MIN(engine_size),
    ROUND(AVG(engine_size), 2),
    MAX(engine_size),
    SUM(CASE WHEN engine_size = 0 THEN 1 ELSE 0 END),
    SUM(CASE WHEN engine_size < 0 THEN 1 ELSE 0 END)
FROM used_cars_deduplicated;
"""

numeric_range_audit = pd.read_sql_query(
    numeric_range_audit_sql,
    conn
)

numeric_range_audit

year_distribution_sql = """
SELECT
    year,
    COUNT(*) AS unique_records,
    SUM(exact_record_count) AS source_records,
    COUNT(DISTINCT manufacturer) AS manufacturer_count
FROM used_cars_deduplicated
GROUP BY year
ORDER BY year DESC;
"""

year_distribution = pd.read_sql_query(
    year_distribution_sql,
    conn
)

year_distribution

future_year_records_sql = """
SELECT
    manufacturer,
    model,
    year,
    price,
    transmission,
    mileage,
    fuel_type,
    tax,
    mpg,
    engine_size,
    source_file,
    exact_record_count
FROM used_cars_deduplicated
WHERE year > 2020
ORDER BY year DESC, manufacturer, model;
"""

future_year_records = pd.read_sql_query(
    future_year_records_sql,
    conn
)

future_year_records

zero_engine_summary_sql = """
SELECT
    manufacturer,
    fuel_type,
    COUNT(*) AS unique_records,
    SUM(exact_record_count) AS source_records,
    COUNT(DISTINCT model) AS model_count,
    MIN(year) AS earliest_year,
    MAX(year) AS latest_year,
    MIN(mpg) AS minimum_mpg,
    MAX(mpg) AS maximum_mpg
FROM used_cars_deduplicated
WHERE engine_size = 0
GROUP BY
    manufacturer,
    fuel_type
ORDER BY
    source_records DESC,
    manufacturer,
    fuel_type;
"""

zero_engine_summary = pd.read_sql_query(
    zero_engine_summary_sql,
    conn
)

zero_engine_summary

extreme_mpg_sql = """
SELECT
    manufacturer,
    model,
    fuel_type,
    mpg,
    MIN(year) AS earliest_year,
    MAX(year) AS latest_year,
    MIN(engine_size) AS minimum_engine_size,
    MAX(engine_size) AS maximum_engine_size,
    COUNT(*) AS unique_records,
    SUM(exact_record_count) AS source_records
FROM used_cars_deduplicated
WHERE mpg < 10
   OR mpg > 150
GROUP BY
    manufacturer,
    model,
    fuel_type,
    mpg
ORDER BY
    mpg DESC,
    manufacturer,
    model;
"""

extreme_mpg = pd.read_sql_query(
    extreme_mpg_sql,
    conn
)

extreme_mpg

remaining_extremes_sql = """
SELECT
    'price_at_or_below_500' AS review_flag,
    COUNT(*) AS unique_records,
    SUM(exact_record_count) AS source_records,
    MIN(year) AS earliest_year,
    MAX(year) AS latest_year,
    MIN(price) AS minimum_price,
    MAX(price) AS maximum_price,
    MIN(mileage) AS minimum_mileage,
    MAX(mileage) AS maximum_mileage
FROM used_cars_deduplicated
WHERE price <= 500

UNION ALL

SELECT
    'price_at_or_above_100000',
    COUNT(*),
    SUM(exact_record_count),
    MIN(year),
    MAX(year),
    MIN(price),
    MAX(price),
    MIN(mileage),
    MAX(mileage)
FROM used_cars_deduplicated
WHERE price >= 100000

UNION ALL

SELECT
    'mileage_at_or_above_200000',
    COUNT(*),
    SUM(exact_record_count),
    MIN(year),
    MAX(year),
    MIN(price),
    MAX(price),
    MIN(mileage),
    MAX(mileage)
FROM used_cars_deduplicated
WHERE mileage >= 200000

UNION ALL

SELECT
    'model_year_before_1980',
    COUNT(*),
    SUM(exact_record_count),
    MIN(year),
    MAX(year),
    MIN(price),
    MAX(price),
    MIN(mileage),
    MAX(mileage)
FROM used_cars_deduplicated
WHERE year < 1980;
"""

remaining_extremes = pd.read_sql_query(
    remaining_extremes_sql,
    conn
)

remaining_extremes

extreme_record_details_sql = """
SELECT
    CASE
        WHEN year < 1980 THEN 'model_year_before_1980'
        WHEN price <= 500 THEN 'price_at_or_below_500'
        WHEN mileage >= 200000 THEN 'mileage_at_or_above_200000'
    END AS review_flag,
    manufacturer,
    model,
    year,
    price,
    transmission,
    mileage,
    fuel_type,
    tax,
    mpg,
    engine_size,
    source_file,
    exact_record_count
FROM used_cars_deduplicated
WHERE year < 1980
   OR price <= 500
   OR mileage >= 200000
ORDER BY
    review_flag,
    year,
    manufacturer,
    model;
"""

extreme_record_details = pd.read_sql_query(
    extreme_record_details_sql,
    conn
)

extreme_record_details

text_hygiene_sql = """
SELECT
    'manufacturer' AS field_name,
    COUNT(*) AS total_records,
    COUNT(DISTINCT manufacturer) AS raw_distinct_values,
    COUNT(DISTINCT LOWER(TRIM(manufacturer))) AS normalized_distinct_values,
    SUM(
        CASE WHEN manufacturer <> TRIM(manufacturer)
        THEN 1 ELSE 0 END
    ) AS records_with_outer_spaces
FROM used_cars_deduplicated

UNION ALL

SELECT
    'model',
    COUNT(*),
    COUNT(DISTINCT model),
    COUNT(DISTINCT LOWER(TRIM(model))),
    SUM(
        CASE WHEN model <> TRIM(model)
        THEN 1 ELSE 0 END
    )
FROM used_cars_deduplicated

UNION ALL

SELECT
    'transmission',
    COUNT(*),
    COUNT(DISTINCT transmission),
    COUNT(DISTINCT LOWER(TRIM(transmission))),
    SUM(
        CASE WHEN transmission <> TRIM(transmission)
        THEN 1 ELSE 0 END
    )
FROM used_cars_deduplicated

UNION ALL

SELECT
    'fuel_type',
    COUNT(*),
    COUNT(DISTINCT fuel_type),
    COUNT(DISTINCT LOWER(TRIM(fuel_type))),
    SUM(
        CASE WHEN fuel_type <> TRIM(fuel_type)
        THEN 1 ELSE 0 END
    )
FROM used_cars_deduplicated

UNION ALL

SELECT
    'source_file',
    COUNT(*),
    COUNT(DISTINCT source_file),
    COUNT(DISTINCT LOWER(TRIM(source_file))),
    SUM(
        CASE WHEN source_file <> TRIM(source_file)
        THEN 1 ELSE 0 END
    )
FROM used_cars_deduplicated;
"""

text_hygiene = pd.read_sql_query(
    text_hygiene_sql,
    conn
)

text_hygiene

category_composition_sql = """
SELECT
    'manufacturer' AS field_name,
    manufacturer AS category_value,
    COUNT(*) AS unique_records,
    SUM(exact_record_count) AS source_records
FROM used_cars_deduplicated
GROUP BY manufacturer

UNION ALL

SELECT
    'transmission',
    transmission,
    COUNT(*),
    SUM(exact_record_count)
FROM used_cars_deduplicated
GROUP BY transmission

UNION ALL

SELECT
    'fuel_type',
    fuel_type,
    COUNT(*),
    SUM(exact_record_count)
FROM used_cars_deduplicated
GROUP BY fuel_type

ORDER BY
    field_name,
    source_records DESC;
"""

category_composition = pd.read_sql_query(
    category_composition_sql,
    conn
)

category_composition

rare_fuel_categories_sql = """
SELECT
    fuel_type,
    manufacturer,
    TRIM(model) AS model,
    COUNT(*) AS unique_records,
    SUM(exact_record_count) AS source_records,
    MIN(year) AS earliest_year,
    MAX(year) AS latest_year,
    MIN(mpg) AS minimum_mpg,
    MAX(mpg) AS maximum_mpg
FROM used_cars_deduplicated
WHERE fuel_type IN ('Other', 'Electric')
GROUP BY
    fuel_type,
    manufacturer,
    TRIM(model)
ORDER BY
    fuel_type,
    source_records DESC,
    manufacturer,
    model;
"""

rare_fuel_categories = pd.read_sql_query(
    rare_fuel_categories_sql,
    conn
)

rare_fuel_categories

other_transmission_records_sql = """
SELECT
    manufacturer,
    TRIM(model) AS model,
    year,
    price,
    transmission,
    mileage,
    fuel_type,
    tax,
    mpg,
    engine_size,
    exact_record_count
FROM used_cars_deduplicated
WHERE transmission = 'Other'
ORDER BY
    manufacturer,
    model,
    year;
"""

other_transmission_records = pd.read_sql_query(
    other_transmission_records_sql,
    conn
)

other_transmission_records

source_mapping_sql = """
SELECT
    source_file,
    COUNT(DISTINCT manufacturer) AS manufacturer_count,
    GROUP_CONCAT(DISTINCT manufacturer) AS manufacturers,
    COUNT(*) AS unique_records,
    SUM(exact_record_count) AS source_records
FROM used_cars_deduplicated
GROUP BY source_file
ORDER BY source_file;
"""

source_mapping = pd.read_sql_query(
    source_mapping_sql,
    conn
)

source_mapping

shared_models_sql = """
SELECT
    TRIM(model) AS model,
    COUNT(DISTINCT manufacturer) AS manufacturer_count,
    GROUP_CONCAT(DISTINCT manufacturer) AS manufacturers,
    COUNT(*) AS unique_records,
    SUM(exact_record_count) AS source_records
FROM used_cars_deduplicated
GROUP BY TRIM(model)
HAVING COUNT(DISTINCT manufacturer) > 1
ORDER BY
    manufacturer_count DESC,
    model;
"""

shared_models = pd.read_sql_query(
    shared_models_sql,
    conn
)

shared_models

create_clean_table_sql = """
DROP TABLE IF EXISTS used_cars_clean;

CREATE TABLE used_cars_clean AS

SELECT
    manufacturer,

    TRIM(model) AS model,

    manufacturer || ' | ' || TRIM(model)
        AS manufacturer_model,

    year AS model_year,

    2020 - year AS approximate_vehicle_age,

    price AS listed_price_gbp,

    transmission AS transmission_raw,

    CASE
        WHEN transmission = 'Other' THEN NULL
        ELSE transmission
    END AS transmission_clean,

    mileage AS mileage_miles,

    fuel_type AS fuel_type_raw,

    CASE
        WHEN fuel_type IN ('Electric', 'Other')
            THEN 'Other / unclear'
        ELSE fuel_type
    END AS fuel_type_analysis,

    tax AS annual_tax_gbp,

    mpg AS mpg_raw,

    CASE
        WHEN mpg < 10 THEN NULL
        ELSE mpg
    END AS mpg_clean,

    CASE
        WHEN mpg > 150 THEN 1
        ELSE 0
    END AS high_mpg_flag,

    CASE
        WHEN mpg < 10 THEN 1
        ELSE 0
    END AS invalid_mpg_flag,

    engine_size AS engine_size_raw,

    CASE
        WHEN engine_size = 0 THEN NULL
        ELSE engine_size
    END AS engine_size_clean,

    CASE
        WHEN engine_size = 0 THEN 1
        ELSE 0
    END AS engine_size_unavailable_flag,

    CASE
        WHEN transmission = 'Other' THEN 1
        ELSE 0
    END AS transmission_unavailable_flag,

    CASE
        WHEN fuel_type IN ('Electric', 'Other') THEN 1
        ELSE 0
    END AS unclear_fuel_type_flag,

    source_file,
    exact_record_count

FROM used_cars_deduplicated

WHERE year BETWEEN 1980 AND 2020;
"""

conn.executescript(create_clean_table_sql)

clean_table_validation_sql = """
SELECT
    COUNT(*) AS clean_unique_records,
    SUM(exact_record_count) AS source_records_represented,

    COUNT(DISTINCT manufacturer) AS manufacturers,
    COUNT(DISTINCT model) AS models,
    COUNT(DISTINCT manufacturer_model) AS manufacturer_models,

    MIN(model_year) AS earliest_model_year,
    MAX(model_year) AS latest_model_year,

    MIN(approximate_vehicle_age) AS minimum_approximate_age,
    MAX(approximate_vehicle_age) AS maximum_approximate_age,

    SUM(invalid_mpg_flag) AS invalid_mpg_records,
    SUM(high_mpg_flag) AS high_mpg_records,
    SUM(engine_size_unavailable_flag)
        AS unavailable_engine_size_records,
    SUM(transmission_unavailable_flag)
        AS unavailable_transmission_records,
    SUM(unclear_fuel_type_flag)
        AS unclear_fuel_type_records

FROM used_cars_clean;
"""

clean_table_validation = pd.read_sql_query(
    clean_table_validation_sql,
    conn
)

clean_table_validation

final_reconciliation_sql = """
WITH population_counts AS (
    SELECT
        (SELECT COUNT(*)
         FROM used_cars) AS staging_rows,

        (SELECT COUNT(*)
         FROM used_cars_deduplicated) AS deduplicated_unique_records,

        (SELECT SUM(exact_record_count)
         FROM used_cars_deduplicated) AS deduplicated_source_records,

        (SELECT COUNT(*)
         FROM used_cars_clean) AS clean_unique_records,

        (SELECT SUM(exact_record_count)
         FROM used_cars_clean) AS clean_source_records,

        (SELECT COUNT(*)
         FROM used_cars_deduplicated
         WHERE year NOT BETWEEN 1980 AND 2020)
            AS invalid_year_unique_records,

        (SELECT SUM(exact_record_count)
         FROM used_cars_deduplicated
         WHERE year NOT BETWEEN 1980 AND 2020)
            AS invalid_year_source_records
)

SELECT
    staging_rows,
    deduplicated_unique_records,

    staging_rows - deduplicated_unique_records
        AS collapsed_excess_duplicates,

    clean_unique_records,

    invalid_year_unique_records,
    invalid_year_source_records,

    clean_source_records,

    staging_rows
        - clean_source_records
        - invalid_year_source_records
        AS reconciliation_difference

FROM population_counts;
"""

final_reconciliation = pd.read_sql_query(
    final_reconciliation_sql,
    conn
)

final_reconciliation

numeric_audit_data = pd.read_sql_query(
    """
    SELECT
        listed_price_gbp,
        mileage_miles,
        annual_tax_gbp,
        mpg_clean,
        engine_size_clean,
        approximate_vehicle_age
    FROM used_cars_clean
    """,
    conn
)

quantile_levels = [
    0.000,
    0.001,
    0.010,
    0.050,
    0.250,
    0.500,
    0.750,
    0.950,
    0.990,
    0.999,
    1.000
]

quantile_labels = [
    "minimum",
    "p0_1",
    "p1",
    "p5",
    "p25",
    "median",
    "p75",
    "p95",
    "p99",
    "p99_9",
    "maximum"
]

quantile_audit = numeric_audit_data.quantile(
    quantile_levels
).T

quantile_audit.columns = quantile_labels

quantile_audit.insert(
    0,
    "non_missing_records",
    numeric_audit_data.notna().sum()
)

quantile_audit.insert(
    1,
    "missing_records",
    numeric_audit_data.isna().sum()
)

quantile_audit = (
    quantile_audit
    .reset_index()
    .rename(columns={"index": "field_name"})
    .round(2)
)

quantile_audit

high_price_model_summary_sql = """
SELECT
    manufacturer,
    model,
    COUNT(*) AS unique_records,
    MIN(model_year) AS earliest_year,
    MAX(model_year) AS latest_year,
    MIN(listed_price_gbp) AS minimum_price,
    MAX(listed_price_gbp) AS maximum_price,
    MIN(mileage_miles) AS minimum_mileage,
    MAX(mileage_miles) AS maximum_mileage
FROM used_cars_clean
WHERE listed_price_gbp >= 100000
GROUP BY
    manufacturer,
    model
ORDER BY
    unique_records DESC,
    maximum_price DESC;
"""

high_price_model_summary = pd.read_sql_query(
    high_price_model_summary_sql,
    conn
)

high_price_model_summary

questionable_high_price_sql = """
SELECT
    manufacturer,
    model,
    model_year,
    listed_price_gbp,
    transmission_raw,
    mileage_miles,
    fuel_type_raw,
    annual_tax_gbp,
    mpg_raw,
    engine_size_raw,
    source_file,
    exact_record_count
FROM used_cars_clean
WHERE
    (
        manufacturer = 'Mercedes-Benz'
        AND model = 'A Class'
        AND listed_price_gbp >= 100000
    )
    OR
    (
        manufacturer = 'BMW'
        AND model = '2 Series'
        AND listed_price_gbp >= 100000
    )
ORDER BY
    manufacturer,
    model,
    listed_price_gbp;
"""

questionable_high_price = pd.read_sql_query(
    questionable_high_price_sql,
    conn
)

questionable_high_price

a_class_engine_comparison_sql = """
SELECT
    engine_size_raw,
    COUNT(*) AS unique_records,
    MIN(model_year) AS earliest_year,
    MAX(model_year) AS latest_year,
    ROUND(AVG(listed_price_gbp), 2) AS average_price,
    MIN(listed_price_gbp) AS minimum_price,
    MAX(listed_price_gbp) AS maximum_price,
    MIN(mpg_raw) AS minimum_mpg,
    MAX(mpg_raw) AS maximum_mpg
FROM used_cars_clean
WHERE manufacturer = 'Mercedes-Benz'
  AND model = 'A Class'
GROUP BY engine_size_raw
ORDER BY engine_size_raw;
"""

a_class_engine_comparison = pd.read_sql_query(
    a_class_engine_comparison_sql,
    conn
)

a_class_engine_comparison

bmw_2_series_comparison_sql = """
SELECT
    model_year,
    listed_price_gbp,
    mileage_miles,
    transmission_raw,
    fuel_type_raw,
    engine_size_raw,
    annual_tax_gbp,
    mpg_raw
FROM used_cars_clean
WHERE manufacturer = 'BMW'
  AND model = '2 Series'
  AND model_year BETWEEN 2014 AND 2016
  AND fuel_type_raw = 'Diesel'
  AND engine_size_raw = 2.0
ORDER BY listed_price_gbp DESC
LIMIT 15;
"""

bmw_2_series_comparison = pd.read_sql_query(
    bmw_2_series_comparison_sql,
    conn
)

bmw_2_series_comparison

large_engine_summary_sql = """
SELECT
    manufacturer,
    model,
    fuel_type_analysis,
    engine_size_clean,
    COUNT(*) AS unique_records,
    MIN(model_year) AS earliest_year,
    MAX(model_year) AS latest_year,
    MIN(listed_price_gbp) AS minimum_price,
    MAX(listed_price_gbp) AS maximum_price
FROM used_cars_clean
WHERE engine_size_clean > 5
GROUP BY
    manufacturer,
    model,
    fuel_type_analysis,
    engine_size_clean
ORDER BY
    engine_size_clean DESC,
    unique_records DESC;
"""

large_engine_summary = pd.read_sql_query(
    large_engine_summary_sql,
    conn
)

large_engine_summary

zero_tax_structure_sql = """
WITH tax_audit_base AS (
    SELECT
        CASE
            WHEN model_year <= 2010 THEN '1996-2010'
            WHEN model_year <= 2016 THEN '2011-2016'
            ELSE '2017-2020'
        END AS model_year_band,

        fuel_type_analysis,
        manufacturer_model,
        annual_tax_gbp,
        mpg_clean

    FROM used_cars_clean
)

SELECT
    model_year_band,
    fuel_type_analysis,

    COUNT(*) AS total_unique_records,

    SUM(
        CASE WHEN annual_tax_gbp = 0
        THEN 1 ELSE 0 END
    ) AS zero_tax_records,

    ROUND(
        100.0 * SUM(
            CASE WHEN annual_tax_gbp = 0
            THEN 1 ELSE 0 END
        ) / COUNT(*),
        2
    ) AS zero_tax_percentage,

    COUNT(
        DISTINCT CASE
            WHEN annual_tax_gbp = 0
            THEN manufacturer_model
        END
    ) AS zero_tax_model_count,

    ROUND(
        AVG(
            CASE
                WHEN annual_tax_gbp = 0
                THEN mpg_clean
            END
        ),
        2
    ) AS average_mpg_for_zero_tax

FROM tax_audit_base

GROUP BY
    model_year_band,
    fuel_type_analysis

HAVING SUM(
    CASE WHEN annual_tax_gbp = 0
    THEN 1 ELSE 0 END
) > 0

ORDER BY
    model_year_band,
    zero_tax_records DESC;
"""

zero_tax_structure = pd.read_sql_query(
    zero_tax_structure_sql,
    conn
)

zero_tax_structure

model_support_audit_sql = """
WITH eligible_records AS (
    SELECT *
    FROM used_cars_clean

    WHERE NOT (
        manufacturer = 'Mercedes-Benz'
        AND model = 'A Class'
        AND engine_size_raw = 4.0
    )

    AND NOT (
        manufacturer = 'BMW'
        AND model = '2 Series'
        AND model_year = 2015
        AND listed_price_gbp = 123456
        AND mileage_miles = 33419
    )
),

model_support AS (
    SELECT
        manufacturer_model,
        COUNT(*) AS unique_records
    FROM eligible_records
    GROUP BY manufacturer_model
),

support_bands AS (
    SELECT
        manufacturer_model,
        unique_records,

        CASE
            WHEN unique_records < 25 THEN '01: fewer than 25'
            WHEN unique_records < 50 THEN '02: 25-49'
            WHEN unique_records < 100 THEN '03: 50-99'
            WHEN unique_records < 250 THEN '04: 100-249'
            WHEN unique_records < 500 THEN '05: 250-499'
            ELSE '06: 500 or more'
        END AS support_band

    FROM model_support
)

SELECT
    support_band,
    COUNT(*) AS manufacturer_model_count,
    SUM(unique_records) AS records_in_band,
    MIN(unique_records) AS smallest_model_group,
    MAX(unique_records) AS largest_model_group,

    ROUND(
        100.0 * SUM(unique_records)
        / (SELECT SUM(unique_records) FROM model_support),
        2
    ) AS percentage_of_eligible_records

FROM support_bands

GROUP BY support_band

ORDER BY support_band;
"""

model_support_audit = pd.read_sql_query(
    model_support_audit_sql,
    conn
)

model_support_audit

model_age_support_sql = """
WITH eligible_records AS (
    SELECT
        *,

        CASE
            WHEN approximate_vehicle_age <= 2 THEN '0-2 years'
            WHEN approximate_vehicle_age <= 5 THEN '3-5 years'
            WHEN approximate_vehicle_age <= 10 THEN '6-10 years'
            ELSE '11+ years'
        END AS age_band

    FROM used_cars_clean

    WHERE NOT (
        manufacturer = 'Mercedes-Benz'
        AND model = 'A Class'
        AND engine_size_raw = 4.0
    )

    AND NOT (
        manufacturer = 'BMW'
        AND model = '2 Series'
        AND model_year = 2015
        AND listed_price_gbp = 123456
        AND mileage_miles = 33419
    )
),

segment_support AS (
    SELECT
        manufacturer_model,
        age_band,
        COUNT(*) AS segment_records
    FROM eligible_records
    GROUP BY
        manufacturer_model,
        age_band
),

support_bands AS (
    SELECT
        manufacturer_model,
        age_band,
        segment_records,

        CASE
            WHEN segment_records < 5 THEN '01: fewer than 5'
            WHEN segment_records < 10 THEN '02: 5-9'
            WHEN segment_records < 25 THEN '03: 10-24'
            WHEN segment_records < 50 THEN '04: 25-49'
            WHEN segment_records < 100 THEN '05: 50-99'
            ELSE '06: 100 or more'
        END AS support_band

    FROM segment_support
)

SELECT
    support_band,
    COUNT(*) AS model_age_segments,
    SUM(segment_records) AS records_in_segments,
    MIN(segment_records) AS smallest_segment,
    MAX(segment_records) AS largest_segment,

    ROUND(
        100.0 * SUM(segment_records)
        / (SELECT SUM(segment_records) FROM segment_support),
        2
    ) AS percentage_of_eligible_records

FROM support_bands

GROUP BY support_band

ORDER BY support_band;
"""

model_age_support = pd.read_sql_query(
    model_age_support_sql,
    conn
)

model_age_support

final_table_sql = """
DROP TABLE IF EXISTS used_cars_analysis;
DROP TABLE IF EXISTS used_cars_clean;

CREATE TABLE used_cars_clean AS

WITH cleaned_records AS (
    SELECT
        manufacturer,

        TRIM(model) AS model,

        manufacturer || ' | ' || TRIM(model)
            AS manufacturer_model,

        year AS model_year,

        2020 - year AS approximate_vehicle_age,

        price AS listed_price_gbp,

        transmission AS transmission_raw,

        CASE
            WHEN transmission = 'Other' THEN NULL
            ELSE transmission
        END AS transmission_clean,

        mileage AS mileage_miles,

        fuel_type AS fuel_type_raw,

        CASE
            WHEN fuel_type IN ('Electric', 'Other')
                THEN 'Other / unclear'
            ELSE fuel_type
        END AS fuel_type_analysis,

        tax AS annual_tax_gbp,

        mpg AS mpg_raw,

        CASE
            WHEN mpg < 10 THEN NULL
            ELSE mpg
        END AS mpg_clean,

        CASE
            WHEN mpg > 150 THEN 1
            ELSE 0
        END AS high_mpg_flag,

        CASE
            WHEN mpg < 10 THEN 1
            ELSE 0
        END AS invalid_mpg_flag,

        engine_size AS engine_size_raw,

        CASE
            WHEN engine_size = 0 THEN NULL
            ELSE engine_size
        END AS engine_size_clean,

        CASE
            WHEN engine_size = 0 THEN 1
            ELSE 0
        END AS engine_size_unavailable_flag,

        CASE
            WHEN transmission = 'Other' THEN 1
            ELSE 0
        END AS transmission_unavailable_flag,

        CASE
            WHEN fuel_type IN ('Electric', 'Other') THEN 1
            ELSE 0
        END AS unclear_fuel_type_flag,

        source_file,
        exact_record_count

    FROM used_cars_deduplicated

    WHERE year BETWEEN 1980 AND 2020
),

quality_flags AS (
    SELECT
        *,

        CASE
            WHEN manufacturer = 'Mercedes-Benz'
             AND model = 'A Class'
             AND engine_size_raw = 4.0
            THEN 1
            ELSE 0
        END AS model_label_unreliable_flag,

        CASE
            WHEN manufacturer = 'BMW'
             AND model = '2 Series'
             AND model_year = 2015
             AND listed_price_gbp = 123456
             AND mileage_miles = 33419
            THEN 1
            ELSE 0
        END AS invalid_price_flag

    FROM cleaned_records
)

SELECT
    *,

    CASE
        WHEN model_label_unreliable_flag = 1
          OR invalid_price_flag = 1
        THEN 0
        ELSE 1
    END AS core_analysis_eligible_flag

FROM quality_flags;


CREATE TABLE used_cars_analysis AS

SELECT *
FROM used_cars_clean
WHERE core_analysis_eligible_flag = 1;
"""

conn.executescript(final_table_sql)

final_table_validation_sql = """
SELECT
    (SELECT COUNT(*)
     FROM used_cars_clean) AS clean_audit_records,

    (SELECT SUM(exact_record_count)
     FROM used_cars_clean) AS clean_source_records,

    (SELECT SUM(model_label_unreliable_flag)
     FROM used_cars_clean) AS unreliable_model_records,

    (SELECT SUM(invalid_price_flag)
     FROM used_cars_clean) AS invalid_price_records,

    (SELECT SUM(
        CASE WHEN core_analysis_eligible_flag = 0
        THEN 1 ELSE 0 END
     )
     FROM used_cars_clean) AS total_ineligible_records,

    (SELECT COUNT(*)
     FROM used_cars_analysis) AS analysis_eligible_records,

    (SELECT COUNT(DISTINCT manufacturer)
     FROM used_cars_analysis) AS eligible_manufacturers,

    (SELECT COUNT(DISTINCT manufacturer_model)
     FROM used_cars_analysis) AS eligible_manufacturer_models,

    (
        (SELECT COUNT(*) FROM used_cars_clean)
        -
        (SELECT COUNT(*) FROM used_cars_analysis)
        -
        (SELECT SUM(
            CASE WHEN core_analysis_eligible_flag = 0
            THEN 1 ELSE 0 END
         ) FROM used_cars_clean)
    ) AS eligibility_reconciliation_difference;
"""

final_table_validation = pd.read_sql_query(
    final_table_validation_sql,
    conn
)

final_table_validation







non_price_fingerprint_summary_sql = """
WITH fingerprint_groups AS (
    SELECT
        manufacturer,
        model,
        model_year,
        transmission_raw,
        mileage_miles,
        fuel_type_raw,
        annual_tax_gbp,
        mpg_raw,
        engine_size_raw,

        COUNT(*) AS unique_price_records,
        COUNT(DISTINCT listed_price_gbp) AS distinct_prices,
        SUM(exact_record_count) AS source_records,

        MIN(listed_price_gbp) AS minimum_price,
        MAX(listed_price_gbp) AS maximum_price

    FROM used_cars_clean

    GROUP BY
        manufacturer,
        model,
        model_year,
        transmission_raw,
        mileage_miles,
        fuel_type_raw,
        annual_tax_gbp,
        mpg_raw,
        engine_size_raw

    HAVING COUNT(DISTINCT listed_price_gbp) > 1
)

SELECT
    COUNT(*) AS fingerprints_with_multiple_prices,

    SUM(unique_price_records)
        AS unique_records_in_fingerprints,

    SUM(source_records)
        AS source_records_in_fingerprints,

    SUM(unique_price_records - 1)
        AS excess_unique_price_records,

    MAX(distinct_prices)
        AS largest_number_of_distinct_prices,

    ROUND(
        AVG(maximum_price - minimum_price),
        2
    ) AS average_price_range,

    MAX(maximum_price - minimum_price)
        AS largest_price_range

FROM fingerprint_groups;
"""

non_price_fingerprint_summary = pd.read_sql_query(
    non_price_fingerprint_summary_sql,
    conn
)

non_price_fingerprint_summary

largest_fingerprint_groups_sql = """
WITH fingerprint_groups AS (
    SELECT
        manufacturer,
        model,
        model_year,
        transmission_raw,
        mileage_miles,
        fuel_type_raw,
        annual_tax_gbp,
        mpg_raw,
        engine_size_raw,

        COUNT(*) AS unique_price_records,
        COUNT(DISTINCT listed_price_gbp) AS distinct_prices,
        SUM(exact_record_count) AS source_records,

        MIN(listed_price_gbp) AS minimum_price,
        MAX(listed_price_gbp) AS maximum_price,
        ROUND(AVG(listed_price_gbp), 2) AS average_price

    FROM used_cars_clean

    GROUP BY
        manufacturer,
        model,
        model_year,
        transmission_raw,
        mileage_miles,
        fuel_type_raw,
        annual_tax_gbp,
        mpg_raw,
        engine_size_raw

    HAVING COUNT(DISTINCT listed_price_gbp) > 1
)

SELECT
    *,
    maximum_price - minimum_price AS price_range
FROM fingerprint_groups

ORDER BY
    distinct_prices DESC,
    price_range DESC

LIMIT 20;
"""

largest_fingerprint_groups = pd.read_sql_query(
    largest_fingerprint_groups_sql,
    conn
)

largest_fingerprint_groups

price_quality_data = pd.read_sql_query(
    """
    SELECT
        manufacturer,
        model,
        manufacturer_model,
        model_year,
        listed_price_gbp,
        mileage_miles,
        transmission_raw,
        fuel_type_raw,
        engine_size_raw
    FROM used_cars_clean
    WHERE core_analysis_eligible_flag = 1
    """,
    conn
)

price_quality_data["group_record_count"] = (
    price_quality_data
    .groupby(["manufacturer_model", "model_year"])
    ["listed_price_gbp"]
    .transform("size")
)

price_quality_data["group_median_price"] = (
    price_quality_data
    .groupby(["manufacturer_model", "model_year"])
    ["listed_price_gbp"]
    .transform("median")
)

price_quality_data["price_to_group_median"] = (
    price_quality_data["listed_price_gbp"]
    / price_quality_data["group_median_price"]
)

price_quality_candidates = price_quality_data[
    (price_quality_data["group_record_count"] >= 25)
    & (
        (price_quality_data["price_to_group_median"] >= 4)
        | (price_quality_data["price_to_group_median"] <= 0.25)
    )
].copy()

price_quality_candidates["deviation_multiple"] = (
    price_quality_candidates["price_to_group_median"]
    .where(
        price_quality_candidates["price_to_group_median"] >= 1,
        1 / price_quality_candidates["price_to_group_median"]
    )
)

price_quality_candidates = (
    price_quality_candidates
    .sort_values("deviation_multiple", ascending=False)
    .round(2)
)

price_quality_candidates

final_quality_update_sql = """
UPDATE used_cars_clean

SET invalid_price_flag =
    CASE
        WHEN (
            manufacturer = 'BMW'
            AND model = '2 Series'
            AND model_year = 2015
            AND listed_price_gbp = 123456
            AND mileage_miles = 33419
        )

        OR (
            manufacturer = 'Hyundai'
            AND model = 'I10'
            AND model_year = 2017
            AND listed_price_gbp = 92000
            AND mileage_miles = 35460
        )

        OR (
            manufacturer = 'Vauxhall'
            AND model = 'Mokka'
            AND model_year = 2016
            AND listed_price_gbp = 52489
            AND mileage_miles = 52489
        )

        OR (
            manufacturer = 'Skoda'
            AND model = 'Karoq'
            AND model_year = 2019
            AND listed_price_gbp = 91874
            AND mileage_miles = 3764
        )

        THEN 1
        ELSE 0
    END;


UPDATE used_cars_clean

SET core_analysis_eligible_flag =
    CASE
        WHEN model_label_unreliable_flag = 1
          OR invalid_price_flag = 1
        THEN 0
        ELSE 1
    END;


DROP TABLE IF EXISTS used_cars_analysis;

CREATE TABLE used_cars_analysis AS

SELECT
    c.*,

    DENSE_RANK() OVER (
        ORDER BY
            manufacturer,
            model,
            model_year,
            transmission_raw,
            mileage_miles,
            fuel_type_raw,
            annual_tax_gbp,
            mpg_raw,
            engine_size_raw
    ) AS non_price_fingerprint_id

FROM used_cars_clean AS c

WHERE core_analysis_eligible_flag = 1;
"""

conn.executescript(final_quality_update_sql)

final_step_3_validation_sql = """
SELECT
    (SELECT COUNT(*)
     FROM used_cars_clean) AS clean_audit_records,

    (SELECT SUM(exact_record_count)
     FROM used_cars_clean) AS clean_source_records,

    (SELECT SUM(model_label_unreliable_flag)
     FROM used_cars_clean) AS unreliable_model_records,

    (SELECT SUM(invalid_price_flag)
     FROM used_cars_clean) AS invalid_price_records,

    (SELECT SUM(
        CASE WHEN core_analysis_eligible_flag = 0
        THEN 1 ELSE 0 END
     )
     FROM used_cars_clean) AS total_ineligible_records,

    (SELECT COUNT(*)
     FROM used_cars_analysis) AS analysis_eligible_records,

    (SELECT COUNT(DISTINCT non_price_fingerprint_id)
     FROM used_cars_analysis) AS eligible_fingerprint_groups,

    (SELECT COUNT(DISTINCT manufacturer)
     FROM used_cars_analysis) AS eligible_manufacturers,

    (SELECT COUNT(DISTINCT manufacturer_model)
     FROM used_cars_analysis) AS eligible_manufacturer_models,

    (
        (SELECT COUNT(*) FROM used_cars_clean)
        -
        (SELECT COUNT(*) FROM used_cars_analysis)
        -
        (SELECT SUM(
            CASE WHEN core_analysis_eligible_flag = 0
            THEN 1 ELSE 0 END
         ) FROM used_cars_clean)
    ) AS eligibility_reconciliation_difference;
"""

final_step_3_validation = pd.read_sql_query(
    final_step_3_validation_sql,
    conn
)

final_step_3_validation

fingerprint_segment_support_sql = """
WITH segment_support AS (
    SELECT
        manufacturer_model,

        CASE
            WHEN approximate_vehicle_age <= 2 THEN '0-2 years'
            WHEN approximate_vehicle_age <= 5 THEN '3-5 years'
            WHEN approximate_vehicle_age <= 10 THEN '6-10 years'
            ELSE '11+ years'
        END AS age_band,

        COUNT(*) AS record_count,

        COUNT(DISTINCT non_price_fingerprint_id)
            AS fingerprint_count

    FROM used_cars_analysis

    GROUP BY
        manufacturer_model,
        age_band
),

support_bands AS (
    SELECT
        manufacturer_model,
        age_band,
        record_count,
        fingerprint_count,

        CASE
            WHEN fingerprint_count < 5 THEN '01: fewer than 5'
            WHEN fingerprint_count < 10 THEN '02: 5-9'
            WHEN fingerprint_count < 25 THEN '03: 10-24'
            WHEN fingerprint_count < 50 THEN '04: 25-49'
            WHEN fingerprint_count < 100 THEN '05: 50-99'
            ELSE '06: 100 or more'
        END AS fingerprint_support_band

    FROM segment_support
)

SELECT
    fingerprint_support_band,

    COUNT(*) AS model_age_segments,

    SUM(record_count) AS records_in_segments,

    SUM(fingerprint_count) AS fingerprints_in_segments,

    MIN(fingerprint_count) AS smallest_fingerprint_group,

    MAX(fingerprint_count) AS largest_fingerprint_group,

    ROUND(
        100.0 * SUM(fingerprint_count)
        / (
            SELECT COUNT(DISTINCT non_price_fingerprint_id)
            FROM used_cars_analysis
        ),
        2
    ) AS percentage_of_eligible_fingerprints

FROM support_bands

GROUP BY fingerprint_support_band

ORDER BY fingerprint_support_band;
"""

fingerprint_segment_support = pd.read_sql_query(
    fingerprint_segment_support_sql,
    conn
)

fingerprint_segment_support

portfolio_overview_sql = """
WITH ordered_prices AS (
    SELECT
        listed_price_gbp,

        ROW_NUMBER() OVER (
            ORDER BY listed_price_gbp
        ) AS price_rank,

        COUNT(*) OVER () AS total_records

    FROM used_cars_analysis
),

price_statistics AS (
    SELECT
        MIN(listed_price_gbp) AS minimum_price,

        MIN(
            CASE
                WHEN price_rank >= 0.25 * total_records
                THEN listed_price_gbp
            END
        ) AS p25_price,

        AVG(
            CASE
                WHEN price_rank IN (
                    (total_records + 1) / 2,
                    (total_records + 2) / 2
                )
                THEN listed_price_gbp
            END
        ) AS median_price,

        ROUND(
            AVG(listed_price_gbp),
            2
        ) AS average_price,

        MIN(
            CASE
                WHEN price_rank >= 0.75 * total_records
                THEN listed_price_gbp
            END
        ) AS p75_price,

        MIN(
            CASE
                WHEN price_rank >= 0.95 * total_records
                THEN listed_price_gbp
            END
        ) AS p95_price,

        MAX(listed_price_gbp) AS maximum_price

    FROM ordered_prices
)

SELECT
    (SELECT COUNT(*)
     FROM used_cars_analysis) AS analysis_records,

    (SELECT SUM(exact_record_count)
     FROM used_cars_analysis) AS source_records_represented,

    (SELECT COUNT(DISTINCT non_price_fingerprint_id)
     FROM used_cars_analysis) AS fingerprint_groups,

    (SELECT COUNT(DISTINCT manufacturer)
     FROM used_cars_analysis) AS manufacturers,

    (SELECT COUNT(DISTINCT manufacturer_model)
     FROM used_cars_analysis) AS manufacturer_models,

    (SELECT MIN(model_year)
     FROM used_cars_analysis) AS earliest_model_year,

    (SELECT MAX(model_year)
     FROM used_cars_analysis) AS latest_model_year,

    minimum_price,
    p25_price,

    ROUND(median_price, 2)
        AS median_price,

    average_price,
    p75_price,
    p95_price,
    maximum_price

FROM price_statistics;
"""

portfolio_overview = pd.read_sql_query(
    portfolio_overview_sql,
    conn
)

portfolio_overview

manufacturer_landscape_sql = """
WITH ranked_manufacturer_prices AS (
    SELECT
        manufacturer,
        manufacturer_model,
        non_price_fingerprint_id,
        listed_price_gbp,

        ROW_NUMBER() OVER (
            PARTITION BY manufacturer
            ORDER BY listed_price_gbp
        ) AS price_rank,

        COUNT(*) OVER (
            PARTITION BY manufacturer
        ) AS manufacturer_records

    FROM used_cars_analysis
),

manufacturer_statistics AS (
    SELECT
        manufacturer,

        COUNT(*) AS analysis_records,

        COUNT(DISTINCT non_price_fingerprint_id)
            AS fingerprint_groups,

        COUNT(DISTINCT manufacturer_model)
            AS model_count,

        MIN(listed_price_gbp)
            AS minimum_price,

        MIN(
            CASE
                WHEN price_rank >= 0.25 * manufacturer_records
                THEN listed_price_gbp
            END
        ) AS p25_price,

        AVG(
            CASE
                WHEN price_rank IN (
                    (manufacturer_records + 1) / 2,
                    (manufacturer_records + 2) / 2
                )
                THEN listed_price_gbp
            END
        ) AS median_price,

        ROUND(
            AVG(listed_price_gbp),
            2
        ) AS average_price,

        MIN(
            CASE
                WHEN price_rank >= 0.75 * manufacturer_records
                THEN listed_price_gbp
            END
        ) AS p75_price,

        MIN(
            CASE
                WHEN price_rank >= 0.95 * manufacturer_records
                THEN listed_price_gbp
            END
        ) AS p95_price,

        MAX(listed_price_gbp)
            AS maximum_price

    FROM ranked_manufacturer_prices

    GROUP BY manufacturer
)

SELECT
    manufacturer,
    analysis_records,

    ROUND(
        100.0 * analysis_records
        / (SELECT COUNT(*) FROM used_cars_analysis),
        2
    ) AS portfolio_record_share_percentage,

    fingerprint_groups,
    model_count,
    minimum_price,
    p25_price,
    ROUND(median_price, 2) AS median_price,
    average_price,
    p75_price,
    p95_price,
    maximum_price

FROM manufacturer_statistics

ORDER BY median_price DESC;
"""

manufacturer_landscape = pd.read_sql_query(
    manufacturer_landscape_sql,
    conn
)

manufacturer_landscape

top_model_landscape_sql = """
WITH ranked_model_prices AS (
    SELECT
        manufacturer,
        model,
        manufacturer_model,
        model_year,
        non_price_fingerprint_id,
        listed_price_gbp,

        ROW_NUMBER() OVER (
            PARTITION BY manufacturer_model
            ORDER BY listed_price_gbp
        ) AS price_rank,

        COUNT(*) OVER (
            PARTITION BY manufacturer_model
        ) AS model_records

    FROM used_cars_analysis
),

model_statistics AS (
    SELECT
        manufacturer,
        model,
        manufacturer_model,

        COUNT(*) AS analysis_records,

        COUNT(DISTINCT non_price_fingerprint_id)
            AS fingerprint_groups,

        MIN(model_year) AS earliest_model_year,
        MAX(model_year) AS latest_model_year,

        MIN(
            CASE
                WHEN price_rank >= 0.25 * model_records
                THEN listed_price_gbp
            END
        ) AS p25_price,

        AVG(
            CASE
                WHEN price_rank IN (
                    (model_records + 1) / 2,
                    (model_records + 2) / 2
                )
                THEN listed_price_gbp
            END
        ) AS median_price,

        MIN(
            CASE
                WHEN price_rank >= 0.75 * model_records
                THEN listed_price_gbp
            END
        ) AS p75_price

    FROM ranked_model_prices

    GROUP BY
        manufacturer,
        model,
        manufacturer_model
)

SELECT
    manufacturer,
    model,
    analysis_records,

    ROUND(
        100.0 * analysis_records
        / (SELECT COUNT(*) FROM used_cars_analysis),
        2
    ) AS portfolio_record_share_percentage,

    fingerprint_groups,
    earliest_model_year,
    latest_model_year,
    p25_price,
    ROUND(median_price, 2) AS median_price,
    p75_price

FROM model_statistics

ORDER BY
    analysis_records DESC

LIMIT 15;
"""

top_model_landscape = pd.read_sql_query(
    top_model_landscape_sql,
    conn
)

top_model_landscape

category_landscape_sql = """
WITH category_records AS (
    SELECT
        'Fuel type' AS category_dimension,
        fuel_type_analysis AS category_value,
        listed_price_gbp,
        non_price_fingerprint_id
    FROM used_cars_analysis

    UNION ALL

    SELECT
        'Transmission',
        COALESCE(
            transmission_clean,
            'Unavailable'
        ),
        listed_price_gbp,
        non_price_fingerprint_id
    FROM used_cars_analysis
),

ranked_category_prices AS (
    SELECT
        category_dimension,
        category_value,
        listed_price_gbp,
        non_price_fingerprint_id,

        ROW_NUMBER() OVER (
            PARTITION BY
                category_dimension,
                category_value
            ORDER BY listed_price_gbp
        ) AS price_rank,

        COUNT(*) OVER (
            PARTITION BY
                category_dimension,
                category_value
        ) AS category_records

    FROM category_records
),

category_statistics AS (
    SELECT
        category_dimension,
        category_value,

        COUNT(*) AS analysis_records,

        COUNT(DISTINCT non_price_fingerprint_id)
            AS fingerprint_groups,

        MIN(
            CASE
                WHEN price_rank >= 0.25 * category_records
                THEN listed_price_gbp
            END
        ) AS p25_price,

        AVG(
            CASE
                WHEN price_rank IN (
                    (category_records + 1) / 2,
                    (category_records + 2) / 2
                )
                THEN listed_price_gbp
            END
        ) AS median_price,

        ROUND(
            AVG(listed_price_gbp),
            2
        ) AS average_price,

        MIN(
            CASE
                WHEN price_rank >= 0.75 * category_records
                THEN listed_price_gbp
            END
        ) AS p75_price

    FROM ranked_category_prices

    GROUP BY
        category_dimension,
        category_value
)

SELECT
    category_dimension,
    category_value,
    analysis_records,

    ROUND(
        100.0 * analysis_records
        / (SELECT COUNT(*) FROM used_cars_analysis),
        2
    ) AS portfolio_share_percentage,

    fingerprint_groups,
    p25_price,
    ROUND(median_price, 2) AS median_price,
    average_price,
    p75_price

FROM category_statistics

ORDER BY
    category_dimension,
    median_price DESC;
"""

category_landscape = pd.read_sql_query(
    category_landscape_sql,
    conn
)

category_landscape

fingerprint_population = pd.read_sql_query(
    """
    SELECT
        non_price_fingerprint_id,
        manufacturer,
        COUNT(*) AS records_in_fingerprint
    FROM used_cars_analysis
    GROUP BY
        non_price_fingerprint_id,
        manufacturer
    """,
    conn
)

test_fingerprints = (
    fingerprint_population
    .groupby(
        "manufacturer",
        group_keys=False
    )
    .sample(
        frac=0.20,
        random_state=42
    )
    ["non_price_fingerprint_id"]
)

fingerprint_population["data_split"] = "train"

fingerprint_population.loc[
    fingerprint_population[
        "non_price_fingerprint_id"
    ].isin(test_fingerprints),
    "data_split"
] = "test"

fingerprint_split_map = fingerprint_population[
    [
        "non_price_fingerprint_id",
        "data_split"
    ]
].copy()

fingerprint_split_map.to_sql(
    "fingerprint_split_map",
    conn,
    index=False,
    if_exists="replace"
)

create_split_table_sql = """
DROP TABLE IF EXISTS used_cars_analysis_split;

CREATE TABLE used_cars_analysis_split AS

SELECT
    a.*,
    s.data_split

FROM used_cars_analysis AS a

INNER JOIN fingerprint_split_map AS s
    ON a.non_price_fingerprint_id
     = s.non_price_fingerprint_id;
"""

conn.executescript(create_split_table_sql)

split_structure_sql = """
WITH overlapping_fingerprints AS (
    SELECT
        non_price_fingerprint_id
    FROM used_cars_analysis_split
    GROUP BY non_price_fingerprint_id
    HAVING COUNT(DISTINCT data_split) > 1
)

SELECT
    data_split,

    COUNT(*) AS analysis_records,

    ROUND(
        100.0 * COUNT(*)
        / (SELECT COUNT(*) FROM used_cars_analysis_split),
        2
    ) AS record_percentage,

    COUNT(DISTINCT non_price_fingerprint_id)
        AS fingerprint_groups,

    ROUND(
        100.0
        * COUNT(DISTINCT non_price_fingerprint_id)
        / (
            SELECT COUNT(DISTINCT non_price_fingerprint_id)
            FROM used_cars_analysis_split
        ),
        2
    ) AS fingerprint_percentage,

    COUNT(DISTINCT manufacturer)
        AS manufacturers,

    COUNT(DISTINCT manufacturer_model)
        AS manufacturer_models,

    (SELECT COUNT(*)
     FROM overlapping_fingerprints)
        AS fingerprints_in_both_splits

FROM used_cars_analysis_split

GROUP BY data_split

ORDER BY data_split;
"""

split_structure = pd.read_sql_query(
    split_structure_sql,
    conn
)

split_structure

split_balance_data = pd.read_sql_query(
    """
    SELECT
        data_split,
        non_price_fingerprint_id,
        listed_price_gbp,
        approximate_vehicle_age,
        mileage_miles
    FROM used_cars_analysis_split
    """,
    conn
)

overall_split_balance = (
    split_balance_data
    .groupby("data_split")
    .agg(
        analysis_records=(
            "listed_price_gbp",
            "size"
        ),
        fingerprint_groups=(
            "non_price_fingerprint_id",
            "nunique"
        ),
        average_price=(
            "listed_price_gbp",
            "mean"
        ),
        median_price=(
            "listed_price_gbp",
            "median"
        ),
        p25_price=(
            "listed_price_gbp",
            lambda x: x.quantile(0.25)
        ),
        p75_price=(
            "listed_price_gbp",
            lambda x: x.quantile(0.75)
        ),
        median_approximate_age=(
            "approximate_vehicle_age",
            "median"
        ),
        median_mileage=(
            "mileage_miles",
            "median"
        )
    )
    .reset_index()
    .round(2)
)

overall_split_balance

categorical_split_balance_sql = """
WITH category_records AS (
    SELECT
        data_split,
        'Manufacturer' AS category_dimension,
        manufacturer AS category_value
    FROM used_cars_analysis_split

    UNION ALL

    SELECT
        data_split,
        'Fuel type',
        fuel_type_analysis
    FROM used_cars_analysis_split

    UNION ALL

    SELECT
        data_split,
        'Transmission',
        COALESCE(
            transmission_clean,
            'Unavailable'
        )
    FROM used_cars_analysis_split
),

category_counts AS (
    SELECT
        category_dimension,
        category_value,

        SUM(
            CASE WHEN data_split = 'train'
            THEN 1 ELSE 0 END
        ) AS train_records,

        SUM(
            CASE WHEN data_split = 'test'
            THEN 1 ELSE 0 END
        ) AS test_records

    FROM category_records

    GROUP BY
        category_dimension,
        category_value
)

SELECT
    category_dimension,
    category_value,
    train_records,
    test_records,

    ROUND(
        100.0 * train_records
        / (SELECT COUNT(*)
           FROM used_cars_analysis_split
           WHERE data_split = 'train'),
        2
    ) AS train_share_percentage,

    ROUND(
        100.0 * test_records
        / (SELECT COUNT(*)
           FROM used_cars_analysis_split
           WHERE data_split = 'test'),
        2
    ) AS test_share_percentage,

    ROUND(
        ABS(
            100.0 * train_records
            / (SELECT COUNT(*)
               FROM used_cars_analysis_split
               WHERE data_split = 'train')
            -
            100.0 * test_records
            / (SELECT COUNT(*)
               FROM used_cars_analysis_split
               WHERE data_split = 'test')
        ),
        2
    ) AS absolute_percentage_point_difference

FROM category_counts

ORDER BY
    category_dimension,
    category_value;
"""

categorical_split_balance = pd.read_sql_query(
    categorical_split_balance_sql,
    conn
)

categorical_split_balance

create_comparable_medians_sql = """
DROP TABLE IF EXISTS comparable_medians_train;

CREATE TABLE comparable_medians_train AS

WITH train_base AS (
    SELECT
        manufacturer,
        manufacturer_model,
        non_price_fingerprint_id,
        listed_price_gbp,

        CASE
            WHEN approximate_vehicle_age <= 2
                THEN '0-2 years'
            WHEN approximate_vehicle_age <= 5
                THEN '3-5 years'
            WHEN approximate_vehicle_age <= 10
                THEN '6-10 years'
            ELSE '11+ years'
        END AS age_band

    FROM used_cars_analysis_split

    WHERE data_split = 'train'
),

segment_rows AS (
    SELECT
        '01_model_age' AS segment_level,
        manufacturer_model || ' || ' || age_band
            AS segment_key,
        non_price_fingerprint_id,
        listed_price_gbp
    FROM train_base

    UNION ALL

    SELECT
        '02_model',
        manufacturer_model,
        non_price_fingerprint_id,
        listed_price_gbp
    FROM train_base

    UNION ALL

    SELECT
        '03_manufacturer_age',
        manufacturer || ' || ' || age_band,
        non_price_fingerprint_id,
        listed_price_gbp
    FROM train_base

    UNION ALL

    SELECT
        '04_manufacturer',
        manufacturer,
        non_price_fingerprint_id,
        listed_price_gbp
    FROM train_base

    UNION ALL

    SELECT
        '05_overall',
        'ALL',
        non_price_fingerprint_id,
        listed_price_gbp
    FROM train_base
),

segment_support AS (
    SELECT
        segment_level,
        segment_key,
        COUNT(*) AS record_support,
        COUNT(DISTINCT non_price_fingerprint_id)
            AS fingerprint_support

    FROM segment_rows

    GROUP BY
        segment_level,
        segment_key
),

ranked_prices AS (
    SELECT
        segment_level,
        segment_key,
        listed_price_gbp,

        ROW_NUMBER() OVER (
            PARTITION BY
                segment_level,
                segment_key
            ORDER BY listed_price_gbp
        ) AS price_rank,

        COUNT(*) OVER (
            PARTITION BY
                segment_level,
                segment_key
        ) AS segment_records

    FROM segment_rows
),

segment_medians AS (
    SELECT
        r.segment_level,
        r.segment_key,

        MAX(s.record_support)
            AS record_support,

        MAX(s.fingerprint_support)
            AS fingerprint_support,

        AVG(
            CASE
                WHEN r.price_rank IN (
                    (r.segment_records + 1) / 2,
                    (r.segment_records + 2) / 2
                )
                THEN r.listed_price_gbp
            END
        ) AS comparable_median_price

    FROM ranked_prices AS r

    INNER JOIN segment_support AS s
        ON r.segment_level = s.segment_level
       AND r.segment_key = s.segment_key

    GROUP BY
        r.segment_level,
        r.segment_key
)

SELECT
    segment_level,
    segment_key,
    record_support,
    fingerprint_support,
    ROUND(
        comparable_median_price,
        2
    ) AS comparable_median_price

FROM segment_medians

WHERE fingerprint_support >= 25;
"""

conn.executescript(create_comparable_medians_sql)

comparable_segment_summary_sql = """
SELECT
    segment_level,
    COUNT(*) AS eligible_segments,
    MIN(fingerprint_support)
        AS minimum_fingerprint_support,
    ROUND(
        AVG(fingerprint_support),
        2
    ) AS average_fingerprint_support,
    MAX(fingerprint_support)
        AS maximum_fingerprint_support,
    MIN(comparable_median_price)
        AS minimum_segment_median,
    MAX(comparable_median_price)
        AS maximum_segment_median

FROM comparable_medians_train

GROUP BY segment_level

ORDER BY segment_level;
"""

comparable_segment_summary = pd.read_sql_query(
    comparable_segment_summary_sql,
    conn
)

comparable_segment_summary

create_test_comparable_predictions_sql = """
DROP TABLE IF EXISTS test_comparable_predictions;

CREATE TABLE test_comparable_predictions AS

WITH test_base AS (
    SELECT
        non_price_fingerprint_id,
        manufacturer,
        model,
        manufacturer_model,
        model_year,
        approximate_vehicle_age,
        mileage_miles,
        fuel_type_analysis,
        transmission_clean,
        engine_size_clean,
        listed_price_gbp
            AS actual_listed_price_gbp,

        CASE
            WHEN approximate_vehicle_age <= 2
                THEN '0-2 years'
            WHEN approximate_vehicle_age <= 5
                THEN '3-5 years'
            WHEN approximate_vehicle_age <= 10
                THEN '6-10 years'
            ELSE '11+ years'
        END AS age_band

    FROM used_cars_analysis_split

    WHERE data_split = 'test'
)

SELECT
    t.*,

    COALESCE(
        model_age.comparable_median_price,
        model_level.comparable_median_price,
        manufacturer_age.comparable_median_price,
        manufacturer_level.comparable_median_price,
        overall_level.comparable_median_price
    ) AS comparable_predicted_price_gbp,

    CASE
        WHEN model_age.comparable_median_price
            IS NOT NULL
            THEN '01_model_age'

        WHEN model_level.comparable_median_price
            IS NOT NULL
            THEN '02_model'

        WHEN manufacturer_age.comparable_median_price
            IS NOT NULL
            THEN '03_manufacturer_age'

        WHEN manufacturer_level.comparable_median_price
            IS NOT NULL
            THEN '04_manufacturer'

        ELSE '05_overall'
    END AS comparable_level_used,

    COALESCE(
        model_age.fingerprint_support,
        model_level.fingerprint_support,
        manufacturer_age.fingerprint_support,
        manufacturer_level.fingerprint_support,
        overall_level.fingerprint_support
    ) AS comparable_fingerprint_support,

    overall_level.comparable_median_price
        AS overall_median_predicted_price_gbp

FROM test_base AS t

LEFT JOIN comparable_medians_train AS model_age
    ON model_age.segment_level = '01_model_age'
   AND model_age.segment_key =
       t.manufacturer_model || ' || ' || t.age_band

LEFT JOIN comparable_medians_train AS model_level
    ON model_level.segment_level = '02_model'
   AND model_level.segment_key =
       t.manufacturer_model

LEFT JOIN comparable_medians_train AS manufacturer_age
    ON manufacturer_age.segment_level =
       '03_manufacturer_age'
   AND manufacturer_age.segment_key =
       t.manufacturer || ' || ' || t.age_band

LEFT JOIN comparable_medians_train AS manufacturer_level
    ON manufacturer_level.segment_level =
       '04_manufacturer'
   AND manufacturer_level.segment_key =
       t.manufacturer

CROSS JOIN comparable_medians_train AS overall_level

WHERE overall_level.segment_level = '05_overall'
  AND overall_level.segment_key = 'ALL';
"""

conn.executescript(
    create_test_comparable_predictions_sql
)

comparable_coverage_sql = """
SELECT
    comparable_level_used,

    COUNT(*) AS test_records,

    ROUND(
        100.0 * COUNT(*)
        / (SELECT COUNT(*)
           FROM test_comparable_predictions),
        2
    ) AS test_record_percentage,

    MIN(comparable_fingerprint_support)
        AS minimum_fingerprint_support,

    ROUND(
        AVG(comparable_fingerprint_support),
        2
    ) AS average_fingerprint_support,

    MAX(comparable_fingerprint_support)
        AS maximum_fingerprint_support,

    SUM(
        CASE
            WHEN comparable_predicted_price_gbp
                IS NULL
            THEN 1
            ELSE 0
        END
    ) AS missing_comparable_predictions

FROM test_comparable_predictions

GROUP BY comparable_level_used

ORDER BY comparable_level_used;
"""

comparable_coverage = pd.read_sql_query(
    comparable_coverage_sql,
    conn
)

comparable_coverage

import numpy as np
import pandas as pd

comparable_test_results = pd.read_sql_query(
    """
    SELECT
        actual_listed_price_gbp,
        overall_median_predicted_price_gbp,
        comparable_predicted_price_gbp
    FROM test_comparable_predictions
    """,
    conn
)

def calculate_pricing_metrics(
    actual,
    predicted,
    benchmark_name
):
    error = predicted - actual
    absolute_error = np.abs(error)

    absolute_percentage_error = (
        absolute_error / actual
    ) * 100

    signed_percentage_error = (
        error / actual
    ) * 100

    return {
        "benchmark": benchmark_name,
        "test_records": len(actual),

        "mae_gbp": np.mean(
            absolute_error
        ),

        "median_absolute_error_gbp": np.median(
            absolute_error
        ),

        "mdape_percentage": np.median(
            absolute_percentage_error
        ),

        "within_10_percent": np.mean(
            absolute_percentage_error <= 10
        ) * 100,

        "within_15_percent": np.mean(
            absolute_percentage_error <= 15
        ) * 100,

        "within_20_percent": np.mean(
            absolute_percentage_error <= 20
        ) * 100,

        "median_signed_percentage_error": np.median(
            signed_percentage_error
        )
    }

benchmark_metrics = pd.DataFrame([
    calculate_pricing_metrics(
        comparable_test_results[
            "actual_listed_price_gbp"
        ],
        comparable_test_results[
            "overall_median_predicted_price_gbp"
        ],
        "Baseline 0: overall median"
    ),

    calculate_pricing_metrics(
        comparable_test_results[
            "actual_listed_price_gbp"
        ],
        comparable_test_results[
            "comparable_predicted_price_gbp"
        ],
        "Baseline 1: hierarchical comparables"
    )
]).round(2)

benchmark_metrics

age_price_schedule_sql = """
WITH ranked_age_prices AS (
    SELECT
        approximate_vehicle_age,
        model_year,
        listed_price_gbp,
        non_price_fingerprint_id,

        ROW_NUMBER() OVER (
            PARTITION BY approximate_vehicle_age
            ORDER BY listed_price_gbp
        ) AS price_rank,

        COUNT(*) OVER (
            PARTITION BY approximate_vehicle_age
        ) AS age_records

    FROM used_cars_analysis
),

age_statistics AS (
    SELECT
        approximate_vehicle_age,
        model_year,

        COUNT(*) AS analysis_records,

        COUNT(DISTINCT non_price_fingerprint_id)
            AS fingerprint_groups,

        MIN(
            CASE
                WHEN price_rank >= 0.25 * age_records
                THEN listed_price_gbp
            END
        ) AS p25_price,

        AVG(
            CASE
                WHEN price_rank IN (
                    (age_records + 1) / 2,
                    (age_records + 2) / 2
                )
                THEN listed_price_gbp
            END
        ) AS median_price,

        ROUND(
            AVG(listed_price_gbp),
            2
        ) AS average_price,

        MIN(
            CASE
                WHEN price_rank >= 0.75 * age_records
                THEN listed_price_gbp
            END
        ) AS p75_price

    FROM ranked_age_prices

    GROUP BY
        approximate_vehicle_age,
        model_year
),

age_zero_reference AS (
    SELECT
        median_price AS reference_median_price
    FROM age_statistics
    WHERE approximate_vehicle_age = 0
)

SELECT
    a.approximate_vehicle_age,
    a.model_year,
    a.analysis_records,
    a.fingerprint_groups,
    a.p25_price,
    ROUND(a.median_price, 2) AS median_price,
    a.average_price,
    a.p75_price,

    ROUND(
        100.0 * a.median_price
        / r.reference_median_price,
        1
    ) AS age_price_index_age_0_equals_100,

    CASE
        WHEN a.fingerprint_groups >= 25
            THEN 'Sufficient support'
        ELSE 'Low support'
    END AS support_status

FROM age_statistics AS a

CROSS JOIN age_zero_reference AS r

ORDER BY a.approximate_vehicle_age;
"""

age_price_schedule = pd.read_sql_query(
    age_price_schedule_sql,
    conn
)

age_price_schedule

mileage_price_schedule_sql = """
WITH mileage_base AS (
    SELECT
        listed_price_gbp,
        approximate_vehicle_age,
        non_price_fingerprint_id,

        CASE
            WHEN mileage_miles < 5000
                THEN '00,000-04,999'
            WHEN mileage_miles < 10000
                THEN '05,000-09,999'
            WHEN mileage_miles < 20000
                THEN '10,000-19,999'
            WHEN mileage_miles < 30000
                THEN '20,000-29,999'
            WHEN mileage_miles < 40000
                THEN '30,000-39,999'
            WHEN mileage_miles < 60000
                THEN '40,000-59,999'
            WHEN mileage_miles < 80000
                THEN '60,000-79,999'
            WHEN mileage_miles < 100000
                THEN '80,000-99,999'
            WHEN mileage_miles < 150000
                THEN '100,000-149,999'
            ELSE '150,000+'
        END AS mileage_band,

        CASE
            WHEN mileage_miles < 5000 THEN 1
            WHEN mileage_miles < 10000 THEN 2
            WHEN mileage_miles < 20000 THEN 3
            WHEN mileage_miles < 30000 THEN 4
            WHEN mileage_miles < 40000 THEN 5
            WHEN mileage_miles < 60000 THEN 6
            WHEN mileage_miles < 80000 THEN 7
            WHEN mileage_miles < 100000 THEN 8
            WHEN mileage_miles < 150000 THEN 9
            ELSE 10
        END AS mileage_band_order

    FROM used_cars_analysis
),

ranked_prices AS (
    SELECT
        *,

        ROW_NUMBER() OVER (
            PARTITION BY mileage_band
            ORDER BY listed_price_gbp
        ) AS price_rank,

        COUNT(*) OVER (
            PARTITION BY mileage_band
        ) AS band_records

    FROM mileage_base
),

price_statistics AS (
    SELECT
        mileage_band,
        mileage_band_order,

        COUNT(*) AS analysis_records,

        COUNT(DISTINCT non_price_fingerprint_id)
            AS fingerprint_groups,

        MIN(
            CASE
                WHEN price_rank >= 0.25 * band_records
                THEN listed_price_gbp
            END
        ) AS p25_price,

        AVG(
            CASE
                WHEN price_rank IN (
                    (band_records + 1) / 2,
                    (band_records + 2) / 2
                )
                THEN listed_price_gbp
            END
        ) AS median_price,

        ROUND(
            AVG(listed_price_gbp),
            2
        ) AS average_price,

        MIN(
            CASE
                WHEN price_rank >= 0.75 * band_records
                THEN listed_price_gbp
            END
        ) AS p75_price

    FROM ranked_prices

    GROUP BY
        mileage_band,
        mileage_band_order
),

ranked_ages AS (
    SELECT
        mileage_band,

        approximate_vehicle_age,

        ROW_NUMBER() OVER (
            PARTITION BY mileage_band
            ORDER BY approximate_vehicle_age
        ) AS age_rank,

        COUNT(*) OVER (
            PARTITION BY mileage_band
        ) AS band_records

    FROM mileage_base
),

age_statistics AS (
    SELECT
        mileage_band,

        AVG(
            CASE
                WHEN age_rank IN (
                    (band_records + 1) / 2,
                    (band_records + 2) / 2
                )
                THEN approximate_vehicle_age
            END
        ) AS median_approximate_age

    FROM ranked_ages

    GROUP BY mileage_band
),

reference_band AS (
    SELECT
        median_price AS reference_median_price
    FROM price_statistics
    WHERE mileage_band_order = 1
)

SELECT
    p.mileage_band,
    p.analysis_records,
    p.fingerprint_groups,

    ROUND(
        a.median_approximate_age,
        1
    ) AS median_approximate_age,

    p.p25_price,
    ROUND(p.median_price, 2) AS median_price,
    p.average_price,
    p.p75_price,

    ROUND(
        100.0 * p.median_price
        / r.reference_median_price,
        1
    ) AS mileage_price_index_lowest_band_equals_100

FROM price_statistics AS p

INNER JOIN age_statistics AS a
    ON p.mileage_band = a.mileage_band

CROSS JOIN reference_band AS r

ORDER BY p.mileage_band_order;
"""

mileage_price_schedule = pd.read_sql_query(
    mileage_price_schedule_sql,
    conn
)

mileage_price_schedule

create_regression_data_sql = """
DROP TABLE IF EXISTS used_cars_regression_data;

CREATE TABLE used_cars_regression_data AS

SELECT
    s.*,

    CASE
        WHEN supported_model.segment_key IS NOT NULL
            THEN s.manufacturer_model
        ELSE s.manufacturer || ' | Other models'
    END AS regression_model_group,

    CASE
        WHEN s.engine_size_clean IS NOT NULL
         AND s.transmission_clean IS NOT NULL
            THEN 1
        ELSE 0
    END AS regression_complete_case_flag,

    s.approximate_vehicle_age
        AS age_years,

    CASE
        WHEN s.approximate_vehicle_age > 1
            THEN s.approximate_vehicle_age - 1
        ELSE 0
    END AS age_after_1_year,

    CASE
        WHEN s.approximate_vehicle_age > 3
            THEN s.approximate_vehicle_age - 3
        ELSE 0
    END AS age_after_3_years,

    CASE
        WHEN s.approximate_vehicle_age > 5
            THEN s.approximate_vehicle_age - 5
        ELSE 0
    END AS age_after_5_years,

    CASE
        WHEN s.approximate_vehicle_age > 10
            THEN s.approximate_vehicle_age - 10
        ELSE 0
    END AS age_after_10_years,

    s.mileage_miles / 10000.0
        AS mileage_10k,

    CASE
        WHEN s.mileage_miles > 10000
            THEN (s.mileage_miles - 10000) / 10000.0
        ELSE 0
    END AS mileage_after_10k,

    CASE
        WHEN s.mileage_miles > 30000
            THEN (s.mileage_miles - 30000) / 10000.0
        ELSE 0
    END AS mileage_after_30k,

    CASE
        WHEN s.mileage_miles > 60000
            THEN (s.mileage_miles - 60000) / 10000.0
        ELSE 0
    END AS mileage_after_60k,

    CASE
        WHEN s.mileage_miles > 100000
            THEN (s.mileage_miles - 100000) / 10000.0
        ELSE 0
    END AS mileage_after_100k

FROM used_cars_analysis_split AS s

LEFT JOIN comparable_medians_train AS supported_model
    ON supported_model.segment_level = '02_model'
   AND supported_model.segment_key =
       s.manufacturer_model;
"""

conn.executescript(create_regression_data_sql)

regression_population_summary_sql = """
SELECT
    data_split,

    COUNT(*) AS total_records,

    SUM(regression_complete_case_flag)
        AS complete_case_records,

    COUNT(*) - SUM(regression_complete_case_flag)
        AS excluded_incomplete_records,

    SUM(
        CASE WHEN engine_size_clean IS NULL
        THEN 1 ELSE 0 END
    ) AS unavailable_engine_size,

    SUM(
        CASE WHEN transmission_clean IS NULL
        THEN 1 ELSE 0 END
    ) AS unavailable_transmission,

    COUNT(DISTINCT regression_model_group)
        AS regression_model_groups,

    COUNT(DISTINCT non_price_fingerprint_id)
        AS fingerprint_groups

FROM used_cars_regression_data

GROUP BY data_split

ORDER BY data_split;
"""

regression_population_summary = pd.read_sql_query(
    regression_population_summary_sql,
    conn
)

regression_population_summary

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

train_model_data = pd.read_sql_query(
    """
    SELECT *
    FROM used_cars_regression_data
    WHERE data_split = 'train'
      AND regression_complete_case_flag = 1
    """,
    conn
)

test_model_data = pd.read_sql_query(
    """
    SELECT
        r.*,
        c.comparable_predicted_price_gbp,
        c.overall_median_predicted_price_gbp
    FROM used_cars_regression_data AS r

    INNER JOIN test_comparable_predictions AS c
        ON r.non_price_fingerprint_id
         = c.non_price_fingerprint_id
       AND r.listed_price_gbp
         = c.actual_listed_price_gbp

    WHERE r.data_split = 'test'
      AND r.regression_complete_case_flag = 1
    """,
    conn
)

model_a1_formula = """
np.log(listed_price_gbp)
~ age_years
+ mileage_10k
+ engine_size_clean
+ C(
    fuel_type_analysis,
    Treatment(reference='Petrol')
  )
+ C(
    transmission_clean,
    Treatment(reference='Manual')
  )
+ C(regression_model_group)
"""

model_a1 = smf.ols(
    formula=model_a1_formula,
    data=train_model_data
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups":
        train_model_data[
            "non_price_fingerprint_id"
        ]
    }
)

model_a1_smearing_factor = np.mean(
    np.exp(model_a1.resid)
)

test_model_data[
    "model_a1_predicted_price_gbp"
] = (
    np.exp(
        model_a1.predict(test_model_data)
    )
    * model_a1_smearing_factor
)

model_a1_comparison = pd.DataFrame([
    calculate_pricing_metrics(
        test_model_data["listed_price_gbp"],
        test_model_data[
            "overall_median_predicted_price_gbp"
        ],
        "Baseline 0: overall median"
    ),

    calculate_pricing_metrics(
        test_model_data["listed_price_gbp"],
        test_model_data[
            "comparable_predicted_price_gbp"
        ],
        "Baseline 1: hierarchical comparables"
    ),

    calculate_pricing_metrics(
        test_model_data["listed_price_gbp"],
        test_model_data[
            "model_a1_predicted_price_gbp"
        ],
        "Model A1: simple log-linear regression"
    )
]).round(2)

print(
    "Training records:",
    int(model_a1.nobs)
)

print(
    "Adjusted R-squared:",
    round(model_a1.rsquared_adj, 4)
)

print(
    "Smearing factor:",
    round(model_a1_smearing_factor, 4)
)

model_a1_comparison

model_a2_formula = """
np.log(listed_price_gbp)
~ age_years
+ age_after_1_year
+ age_after_3_years
+ age_after_5_years
+ age_after_10_years
+ mileage_10k
+ mileage_after_10k
+ mileage_after_30k
+ mileage_after_60k
+ mileage_after_100k
+ engine_size_clean
+ C(
    fuel_type_analysis,
    Treatment(reference='Petrol')
  )
+ C(
    transmission_clean,
    Treatment(reference='Manual')
  )
+ C(regression_model_group)
"""

model_a2 = smf.ols(
    formula=model_a2_formula,
    data=train_model_data
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups":
        train_model_data[
            "non_price_fingerprint_id"
        ]
    }
)

model_a2_smearing_factor = np.mean(
    np.exp(model_a2.resid)
)

test_model_data[
    "model_a2_predicted_price_gbp"
] = (
    np.exp(
        model_a2.predict(test_model_data)
    )
    * model_a2_smearing_factor
)

model_a2_comparison = pd.DataFrame([
    calculate_pricing_metrics(
        test_model_data["listed_price_gbp"],
        test_model_data[
            "comparable_predicted_price_gbp"
        ],
        "Baseline 1: hierarchical comparables"
    ),

    calculate_pricing_metrics(
        test_model_data["listed_price_gbp"],
        test_model_data[
            "model_a1_predicted_price_gbp"
        ],
        "Model A1: simple log-linear"
    ),

    calculate_pricing_metrics(
        test_model_data["listed_price_gbp"],
        test_model_data[
            "model_a2_predicted_price_gbp"
        ],
        "Model A2: piecewise nonlinear"
    )
]).round(2)

print(
    "A2 training records:",
    int(model_a2.nobs)
)

print(
    "A2 adjusted R-squared:",
    round(model_a2.rsquared_adj, 4)
)

print(
    "A2 smearing factor:",
    round(model_a2_smearing_factor, 4)
)

model_a2_comparison

create_typical_mileage_sql = """
DROP TABLE IF EXISTS typical_mileage_by_age_train;

CREATE TABLE typical_mileage_by_age_train AS

WITH fingerprint_age_mileage AS (
    SELECT DISTINCT
        non_price_fingerprint_id,
        age_years,
        mileage_miles

    FROM used_cars_regression_data

    WHERE data_split = 'train'
      AND regression_complete_case_flag = 1
),

ranked_mileage AS (
    SELECT
        age_years,
        mileage_miles,

        ROW_NUMBER() OVER (
            PARTITION BY age_years
            ORDER BY mileage_miles
        ) AS mileage_rank,

        COUNT(*) OVER (
            PARTITION BY age_years
        ) AS age_fingerprints

    FROM fingerprint_age_mileage
)

SELECT
    age_years,

    MAX(age_fingerprints)
        AS fingerprint_support,

    ROUND(
        AVG(
            CASE
                WHEN mileage_rank IN (
                    (age_fingerprints + 1) / 2,
                    (age_fingerprints + 2) / 2
                )
                THEN mileage_miles
            END
        ),
        2
    ) AS typical_mileage_miles

FROM ranked_mileage

GROUP BY age_years;
"""

conn.executescript(create_typical_mileage_sql)



typical_mileage_by_age = pd.read_sql_query(
    """
    SELECT
        age_years,
        fingerprint_support,
        typical_mileage_miles
    FROM typical_mileage_by_age_train
    ORDER BY age_years
    """,
    conn
)

typical_mileage_by_age

create_excess_mileage_data_sql = """
DROP TABLE IF EXISTS typical_mileage_reference_train;
DROP TABLE IF EXISTS used_cars_regression_excess;

CREATE TABLE typical_mileage_reference_train AS

WITH fingerprint_age_mileage AS (
    SELECT DISTINCT
        non_price_fingerprint_id,

        CASE
            WHEN age_years >= 19 THEN 19
            ELSE age_years
        END AS age_reference_group,

        mileage_miles

    FROM used_cars_regression_data

    WHERE data_split = 'train'
      AND regression_complete_case_flag = 1
),

ranked_mileage AS (
    SELECT
        age_reference_group,
        mileage_miles,

        ROW_NUMBER() OVER (
            PARTITION BY age_reference_group
            ORDER BY mileage_miles
        ) AS mileage_rank,

        COUNT(*) OVER (
            PARTITION BY age_reference_group
        ) AS reference_fingerprints

    FROM fingerprint_age_mileage
)

SELECT
    age_reference_group,

    MAX(reference_fingerprints)
        AS fingerprint_support,

    ROUND(
        AVG(
            CASE
                WHEN mileage_rank IN (
                    (reference_fingerprints + 1) / 2,
                    (reference_fingerprints + 2) / 2
                )
                THEN mileage_miles
            END
        ),
        2
    ) AS typical_mileage_miles

FROM ranked_mileage

GROUP BY age_reference_group;


CREATE TABLE used_cars_regression_excess AS

SELECT
    r.*,

    t.typical_mileage_miles,

    (
        r.mileage_miles
        - t.typical_mileage_miles
    ) / 10000.0 AS excess_mileage_10k

FROM used_cars_regression_data AS r

INNER JOIN typical_mileage_reference_train AS t
    ON t.age_reference_group =
       CASE
           WHEN r.age_years >= 19 THEN 19
           ELSE r.age_years
       END;
"""

conn.executescript(create_excess_mileage_data_sql)

train_excess_data = pd.read_sql_query(
    """
    SELECT *
    FROM used_cars_regression_excess
    WHERE data_split = 'train'
      AND regression_complete_case_flag = 1
    """,
    conn
)

test_excess_data = pd.read_sql_query(
    """
    SELECT
        e.*,
        c.comparable_predicted_price_gbp,
        c.overall_median_predicted_price_gbp
    FROM used_cars_regression_excess AS e

    INNER JOIN test_comparable_predictions AS c
        ON e.non_price_fingerprint_id
         = c.non_price_fingerprint_id
       AND e.listed_price_gbp
         = c.actual_listed_price_gbp

    WHERE e.data_split = 'test'
      AND e.regression_complete_case_flag = 1
    """,
    conn
)

test_excess_data = test_excess_data.merge(
    test_model_data[
        [
            "non_price_fingerprint_id",
            "listed_price_gbp",
            "model_a1_predicted_price_gbp"
        ]
    ],
    on=[
        "non_price_fingerprint_id",
        "listed_price_gbp"
    ],
    how="left",
    validate="one_to_one"
)

model_a3_formula = """
np.log(listed_price_gbp)
~ age_years
+ excess_mileage_10k
+ engine_size_clean
+ C(
    fuel_type_analysis,
    Treatment(reference='Petrol')
  )
+ C(
    transmission_clean,
    Treatment(reference='Manual')
  )
+ C(regression_model_group)
"""

model_a3 = smf.ols(
    formula=model_a3_formula,
    data=train_excess_data
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups":
        train_excess_data[
            "non_price_fingerprint_id"
        ]
    }
)

model_a3_smearing_factor = np.mean(
    np.exp(model_a3.resid)
)

test_excess_data[
    "model_a3_predicted_price_gbp"
] = (
    np.exp(
        model_a3.predict(test_excess_data)
    )
    * model_a3_smearing_factor
)

model_a3_comparison = pd.DataFrame([
    calculate_pricing_metrics(
        test_excess_data["listed_price_gbp"],
        test_excess_data[
            "comparable_predicted_price_gbp"
        ],
        "Baseline 1: hierarchical comparables"
    ),

    calculate_pricing_metrics(
        test_excess_data["listed_price_gbp"],
        test_excess_data[
            "model_a1_predicted_price_gbp"
        ],
        "Model A1: direct mileage"
    ),

    calculate_pricing_metrics(
        test_excess_data["listed_price_gbp"],
        test_excess_data[
            "model_a3_predicted_price_gbp"
        ],
        "Model A3: excess mileage for age"
    )
]).round(2)

print(
    "A3 adjusted R-squared:",
    round(model_a3.rsquared_adj, 4)
)

print(
    "A3 smearing factor:",
    round(model_a3_smearing_factor, 4)
)

model_a3_comparison

segment_validation_data = test_excess_data.copy()

segment_validation_data["price_band"] = pd.cut(
    segment_validation_data["listed_price_gbp"],
    bins=[
        0,
        10000,
        20000,
        30000,
        50000,
        np.inf
    ],
    labels=[
        "Under £10k",
        "£10k-£20k",
        "£20k-£30k",
        "£30k-£50k",
        "£50k+"
    ],
    right=False
)

segment_validation_data["age_band"] = pd.cut(
    segment_validation_data[
        "approximate_vehicle_age"
    ],
    bins=[
        -1,
        2,
        5,
        10,
        np.inf
    ],
    labels=[
        "0-2 years",
        "3-5 years",
        "6-10 years",
        "11+ years"
    ]
)

def segment_metric_row(
    frame,
    dimension,
    segment
):
    actual = frame["listed_price_gbp"]

    comparable_prediction = frame[
        "comparable_predicted_price_gbp"
    ]

    model_prediction = frame[
        "model_a3_predicted_price_gbp"
    ]

    comparable_ape = (
        np.abs(
            comparable_prediction - actual
        ) / actual
    ) * 100

    model_error = model_prediction - actual

    model_absolute_error = np.abs(
        model_error
    )

    model_ape = (
        model_absolute_error / actual
    ) * 100

    model_signed_percentage_error = (
        model_error / actual
    ) * 100

    return {
        "dimension": dimension,
        "segment": str(segment),
        "test_records": len(frame),

        "comparable_mdape": np.median(
            comparable_ape
        ),

        "model_a3_mdape": np.median(
            model_ape
        ),

        "mdape_improvement_points": (
            np.median(comparable_ape)
            - np.median(model_ape)
        ),

        "model_a3_mae_gbp": np.mean(
            model_absolute_error
        ),

        "model_a3_within_20_percent": np.mean(
            model_ape <= 20
        ) * 100,

        "model_a3_median_signed_error": np.median(
            model_signed_percentage_error
        )
    }


segment_validation_rows = []

segment_definitions = {
    "Price band": "price_band",
    "Manufacturer": "manufacturer",
    "Age band": "age_band"
}

for dimension, column in segment_definitions.items():
    for segment, frame in (
        segment_validation_data
        .groupby(
            column,
            observed=True
        )
    ):
        segment_validation_rows.append(
            segment_metric_row(
                frame,
                dimension,
                segment
            )
        )

segment_validation = (
    pd.DataFrame(segment_validation_rows)
    .round(2)
)

segment_validation

model_segment_comparison_data = (
    test_excess_data
    .merge(
        test_model_data[
            [
                "non_price_fingerprint_id",
                "listed_price_gbp",
                "model_a2_predicted_price_gbp"
            ]
        ],
        on=[
            "non_price_fingerprint_id",
            "listed_price_gbp"
        ],
        how="left",
        validate="one_to_one"
    )
)

model_segment_comparison_data["price_band"] = pd.cut(
    model_segment_comparison_data[
        "listed_price_gbp"
    ],
    bins=[
        0,
        10000,
        20000,
        30000,
        50000,
        np.inf
    ],
    labels=[
        "Under £10k",
        "£10k-£20k",
        "£20k-£30k",
        "£30k-£50k",
        "£50k+"
    ],
    right=False
)

model_segment_comparison_data["age_band"] = pd.cut(
    model_segment_comparison_data[
        "approximate_vehicle_age"
    ],
    bins=[
        -1,
        2,
        5,
        10,
        np.inf
    ],
    labels=[
        "0-2 years",
        "3-5 years",
        "6-10 years",
        "11+ years"
    ]
)

prediction_columns = {
    "a1": "model_a1_predicted_price_gbp",
    "a2": "model_a2_predicted_price_gbp",
    "a3": "model_a3_predicted_price_gbp"
}

segment_model_rows = []

for dimension, column in {
    "Price band": "price_band",
    "Age band": "age_band"
}.items():

    for segment, frame in (
        model_segment_comparison_data
        .groupby(
            column,
            observed=True
        )
    ):
        row = {
            "dimension": dimension,
            "segment": str(segment),
            "test_records": len(frame)
        }

        actual = frame["listed_price_gbp"]

        for model_name, prediction_column in (
            prediction_columns.items()
        ):
            prediction = frame[prediction_column]

            error = prediction - actual

            absolute_percentage_error = (
                np.abs(error) / actual
            ) * 100

            signed_percentage_error = (
                error / actual
            ) * 100

            row[
                f"{model_name}_mdape"
            ] = np.median(
                absolute_percentage_error
            )

            row[
                f"{model_name}_within_20"
            ] = np.mean(
                absolute_percentage_error <= 20
            ) * 100

            row[
                f"{model_name}_median_bias"
            ] = np.median(
                signed_percentage_error
            )

        segment_model_rows.append(row)

segment_model_comparison = (
    pd.DataFrame(segment_model_rows)
    .round(2)
)

segment_model_comparison

model_b1_formula = """
listed_price_gbp
~ age_years
+ age_after_1_year
+ age_after_3_years
+ age_after_5_years
+ age_after_10_years
+ mileage_10k
+ mileage_after_10k
+ mileage_after_30k
+ mileage_after_60k
+ mileage_after_100k
+ engine_size_clean
+ C(
    fuel_type_analysis,
    Treatment(reference='Petrol')
  )
+ C(
    transmission_clean,
    Treatment(reference='Manual')
  )
+ C(regression_model_group)
"""

model_b1 = smf.ols(
    formula=model_b1_formula,
    data=train_model_data
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups":
        train_model_data[
            "non_price_fingerprint_id"
        ]
    }
)

test_model_data[
    "model_b1_predicted_price_gbp"
] = model_b1.predict(
    test_model_data
)

model_b1_comparison = pd.DataFrame([
    calculate_pricing_metrics(
        test_model_data["listed_price_gbp"],
        test_model_data[
            "comparable_predicted_price_gbp"
        ],
        "Baseline 1: hierarchical comparables"
    ),

    calculate_pricing_metrics(
        test_model_data["listed_price_gbp"],
        test_model_data[
            "model_a2_predicted_price_gbp"
        ],
        "Model A2: piecewise log price"
    ),

    calculate_pricing_metrics(
        test_model_data["listed_price_gbp"],
        test_model_data[
            "model_b1_predicted_price_gbp"
        ],
        "Model B1: piecewise raw price"
    )
]).round(2)

print(
    "B1 training records:",
    int(model_b1.nobs)
)

print(
    "B1 adjusted R-squared:",
    round(model_b1.rsquared_adj, 4)
)

print(
    "Negative test predictions:",
    int(
        (
            test_model_data[
                "model_b1_predicted_price_gbp"
            ] < 0
        ).sum()
    )
)

model_b1_comparison

a2_calibration_data = test_model_data[
    [
        "listed_price_gbp",
        "model_a2_predicted_price_gbp"
    ]
].copy()

a2_calibration_data[
    "predicted_price_decile"
] = pd.qcut(
    a2_calibration_data[
        "model_a2_predicted_price_gbp"
    ],
    q=10,
    labels=[
        "D1 lowest",
        "D2",
        "D3",
        "D4",
        "D5",
        "D6",
        "D7",
        "D8",
        "D9",
        "D10 highest"
    ]
)

calibration_rows = []

for decile, frame in (
    a2_calibration_data
    .groupby(
        "predicted_price_decile",
        observed=True
    )
):
    actual = frame["listed_price_gbp"]

    predicted = frame[
        "model_a2_predicted_price_gbp"
    ]

    error = predicted - actual

    absolute_percentage_error = (
        np.abs(error) / actual
    ) * 100

    signed_percentage_error = (
        error / actual
    ) * 100

    calibration_rows.append({
        "predicted_price_decile": str(decile),
        "test_records": len(frame),

        "median_actual_price": np.median(
            actual
        ),

        "median_predicted_price": np.median(
            predicted
        ),

        "predicted_to_actual_median_ratio": (
            np.median(predicted)
            / np.median(actual)
        ),

        "mdape_percentage": np.median(
            absolute_percentage_error
        ),

        "median_signed_error_percentage": np.median(
            signed_percentage_error
        ),

        "mae_gbp": np.mean(
            np.abs(error)
        )
    })

a2_calibration_by_decile = (
    pd.DataFrame(calibration_rows)
    .round(2)
)

a2_calibration_by_decile

age_mileage_relationship = pd.DataFrame([
    {
        "comparison":
            "Age vs direct mileage",
        "pearson_correlation":
            train_model_data[
                ["age_years", "mileage_10k"]
            ].corr(method="pearson").iloc[0, 1],
        "spearman_correlation":
            train_model_data[
                ["age_years", "mileage_10k"]
            ].corr(method="spearman").iloc[0, 1]
    },

    {
        "comparison":
            "Age vs excess mileage",
        "pearson_correlation":
            train_excess_data[
                ["age_years", "excess_mileage_10k"]
            ].corr(method="pearson").iloc[0, 1],
        "spearman_correlation":
            train_excess_data[
                ["age_years", "excess_mileage_10k"]
            ].corr(method="spearman").iloc[0, 1]
    }
]).round(3)

age_mileage_relationship

import statsmodels.api as sm

from statsmodels.stats.outliers_influence import (
    variance_inflation_factor
)

def calculate_vif(
    data,
    variable_names,
    specification
):
    vif_data = (
        data[variable_names]
        .dropna()
        .astype(float)
    )

    vif_matrix = sm.add_constant(
        vif_data
    )

    rows = []

    for index, column in enumerate(
        vif_matrix.columns
    ):
        if column == "const":
            continue

        rows.append({
            "specification": specification,
            "variable": column,
            "vif": variance_inflation_factor(
                vif_matrix.values,
                index
            )
        })

    return rows


vif_rows = []

vif_rows.extend(
    calculate_vif(
        train_model_data,
        [
            "age_years",
            "mileage_10k",
            "engine_size_clean"
        ],
        "Direct mileage"
    )
)

vif_rows.extend(
    calculate_vif(
        train_excess_data,
        [
            "age_years",
            "excess_mileage_10k",
            "engine_size_clean"
        ],
        "Excess mileage"
    )
)

vif_comparison = (
    pd.DataFrame(vif_rows)
    .round(2)
)

vif_comparison

a2_terms = list(
    model_a2.params.index
)

a2_coefficients = np.asarray(
    model_a2.params
)

a2_confidence_intervals = np.asarray(
    model_a2.conf_int()
)

feature_effect_rows = []

for index, term in enumerate(a2_terms):

    include_term = (
        term == "engine_size_clean"
        or "fuel_type_analysis" in term
        or "transmission_clean" in term
    )

    if not include_term:
        continue

    coefficient = a2_coefficients[index]
    lower_coefficient = (
        a2_confidence_intervals[index, 0]
    )
    upper_coefficient = (
        a2_confidence_intervals[index, 1]
    )

    if term == "engine_size_clean":
        feature_label = (
            "Reported engine size: +0.5 unit"
        )
        reference_label = (
            "Same observed attributes"
        )
        effect_scale = 0.5

    elif "fuel_type_analysis" in term:
        category = (
            term
            .split("[T.")[1]
            .rstrip("]")
        )

        feature_label = (
            f"Fuel type: {category}"
        )
        reference_label = "Petrol"
        effect_scale = 1.0

    else:
        category = (
            term
            .split("[T.")[1]
            .rstrip("]")
        )

        feature_label = (
            f"Transmission: {category}"
        )
        reference_label = "Manual"
        effect_scale = 1.0

    feature_effect_rows.append({
        "feature_comparison": feature_label,
        "reference": reference_label,

        "conditional_price_difference_pct":
            (
                np.exp(
                    coefficient * effect_scale
                ) - 1
            ) * 100,

        "confidence_interval_low_pct":
            (
                np.exp(
                    lower_coefficient
                    * effect_scale
                ) - 1
            ) * 100,

        "confidence_interval_high_pct":
            (
                np.exp(
                    upper_coefficient
                    * effect_scale
                ) - 1
            ) * 100
    })

a2_feature_effects = (
    pd.DataFrame(feature_effect_rows)
    .round(2)
)

a2_feature_effects

import numpy as np
import pandas as pd

params = model_a2.params
covariance = model_a2.cov_params()

required_terms = [
    "age_years",
    "age_after_1_year",
    "age_after_3_years",
    "age_after_5_years",
    "age_after_10_years",
    "mileage_10k",
    "mileage_after_10k",
    "mileage_after_30k",
    "mileage_after_60k",
    "mileage_after_100k"
]

missing_terms = [
    term for term in required_terms
    if term not in params.index
]

if missing_terms:
    raise KeyError(
        f"Missing A2 coefficients: {missing_terms}"
    )


def calculate_index_and_ci(term_values):
    """
    Calculate the conditional price index and clustered 95% CI
    relative to the zero-value reference point.
    """
    contrast = pd.Series(0.0, index=params.index)

    for term, value in term_values.items():
        contrast.loc[term] = value

    log_difference = float(contrast @ params)

    variance = float(
        contrast.to_numpy()
        @ covariance.to_numpy()
        @ contrast.to_numpy()
    )

    standard_error = np.sqrt(max(variance, 0))

    price_index = 100 * np.exp(log_difference)
    ci_low = 100 * np.exp(
        log_difference - 1.96 * standard_error
    )
    ci_high = 100 * np.exp(
        log_difference + 1.96 * standard_error
    )

    return price_index, ci_low, ci_high


# ---------------------------------------------------------
# Adjusted age relationship
# Mileage and other included attributes are held constant.
# Ages 19+ are excluded because of low support.
# ---------------------------------------------------------

age_results = []

for age in range(0, 19):
    age_terms = {
        "age_years": age,
        "age_after_1_year": max(age - 1, 0),
        "age_after_3_years": max(age - 3, 0),
        "age_after_5_years": max(age - 5, 0),
        "age_after_10_years": max(age - 10, 0)
    }

    price_index, ci_low, ci_high = calculate_index_and_ci(
        age_terms
    )

    age_results.append({
        "approximate_vehicle_age": age,
        "model_year": 2020 - age,
        "adjusted_price_index_age_0_equals_100": price_index,
        "confidence_interval_low": ci_low,
        "confidence_interval_high": ci_high
    })

adjusted_age_curve = pd.DataFrame(age_results).round(1)


# ---------------------------------------------------------
# Adjusted mileage relationship
# Age and other included attributes are held constant.
# ---------------------------------------------------------

mileage_points = [
    0,
    5000,
    10000,
    20000,
    30000,
    40000,
    60000,
    80000,
    100000,
    150000
]

mileage_results = []

for mileage in mileage_points:
    mileage_10k = mileage / 10000

    mileage_terms = {
        "mileage_10k": mileage_10k,
        "mileage_after_10k": max(mileage_10k - 1, 0),
        "mileage_after_30k": max(mileage_10k - 3, 0),
        "mileage_after_60k": max(mileage_10k - 6, 0),
        "mileage_after_100k": max(mileage_10k - 10, 0)
    }

    price_index, ci_low, ci_high = calculate_index_and_ci(
        mileage_terms
    )

    mileage_results.append({
        "mileage_miles": mileage,
        "adjusted_price_index_0_miles_equals_100": price_index,
        "confidence_interval_low": ci_low,
        "confidence_interval_high": ci_high
    })

adjusted_mileage_curve = pd.DataFrame(
    mileage_results
).round(1)


print("Adjusted age relationship")
display(adjusted_age_curve)

print("Adjusted mileage relationship")
display(adjusted_mileage_curve)

import numpy as np
import pandas as pd

params = model_a2.params
covariance = model_a2.cov_params()

# Retrieve the complete-case training data used by Model A2
model_frame = model_a2.model.data.frame.copy()

required_columns = [
    "regression_model_group",
    "non_price_fingerprint_id"
]

missing_columns = [
    column for column in required_columns
    if column not in model_frame.columns
]

if missing_columns:
    raise KeyError(
        f"Required columns not found in the A2 training data: "
        f"{missing_columns}"
    )


# ---------------------------------------------------------
# Fingerprint support for each regression model group
# ---------------------------------------------------------

model_support = (
    model_frame
    .groupby("regression_model_group")
    ["non_price_fingerprint_id"]
    .nunique()
    .rename("fingerprint_support")
    .reset_index()
)

model_support["manufacturer"] = (
    model_support["regression_model_group"]
    .str.split(" | ", n=1, regex=False)
    .str[0]
)


parameter_names = list(params.index)
parameter_index = {
    name: position
    for position, name in enumerate(parameter_names)
}


def model_group_vector(model_group):
    """
    Return the coefficient-design vector for a model group.
    The omitted reference model receives a zero vector.
    """
    vector = np.zeros(len(params))

    coefficient_name = (
        f"C(regression_model_group)[T.{model_group}]"
    )

    if coefficient_name in parameter_index:
        vector[parameter_index[coefficient_name]] = 1.0

    return vector


# Add the design vector for every model group
model_support["design_vector"] = (
    model_support["regression_model_group"]
    .apply(model_group_vector)
)


# ---------------------------------------------------------
# Portfolio reference:
# fingerprint-weighted average model effect
# ---------------------------------------------------------

total_support = model_support["fingerprint_support"].sum()

portfolio_vector = sum(
    row["design_vector"] * row["fingerprint_support"]
    for _, row in model_support.iterrows()
) / total_support


def index_from_contrast(contrast):
    """
    Convert a log-price contrast into an index where the
    fingerprint-weighted portfolio average equals 100.
    """
    estimate = float(contrast @ params.to_numpy())

    variance = float(
        contrast
        @ covariance.to_numpy()
        @ contrast
    )

    standard_error = np.sqrt(max(variance, 0))

    index_value = 100 * np.exp(estimate)
    ci_low = 100 * np.exp(
        estimate - 1.96 * standard_error
    )
    ci_high = 100 * np.exp(
        estimate + 1.96 * standard_error
    )

    return index_value, ci_low, ci_high


# ---------------------------------------------------------
# Manufacturer-level conditional price indices
# ---------------------------------------------------------

manufacturer_results = []

for manufacturer, group in model_support.groupby("manufacturer"):
    manufacturer_support = group["fingerprint_support"].sum()

    manufacturer_vector = sum(
        row["design_vector"] * row["fingerprint_support"]
        for _, row in group.iterrows()
    ) / manufacturer_support

    contrast = manufacturer_vector - portfolio_vector

    index_value, ci_low, ci_high = index_from_contrast(
        contrast
    )

    manufacturer_results.append({
        "manufacturer": manufacturer,
        "fingerprint_support": manufacturer_support,
        "named_and_pooled_model_groups": len(group),
        "conditional_price_index_portfolio_equals_100":
            index_value,
        "confidence_interval_low": ci_low,
        "confidence_interval_high": ci_high
    })

manufacturer_price_indices = (
    pd.DataFrame(manufacturer_results)
    .sort_values(
        "conditional_price_index_portfolio_equals_100",
        ascending=False
    )
    .round(1)
)


# ---------------------------------------------------------
# Named-model conditional price indices
# Only show named models with at least 100 fingerprints.
# Exclude pooled "Other models" groups from the headline table.
# ---------------------------------------------------------

model_results = []

for _, row in model_support.iterrows():
    model_group = row["regression_model_group"]

    if (
        row["fingerprint_support"] < 100
        or model_group.endswith("Other models")
    ):
        continue

    contrast = row["design_vector"] - portfolio_vector

    index_value, ci_low, ci_high = index_from_contrast(
        contrast
    )

    model_results.append({
        "manufacturer_model": model_group,
        "fingerprint_support": row["fingerprint_support"],
        "conditional_price_index_portfolio_equals_100":
            index_value,
        "confidence_interval_low": ci_low,
        "confidence_interval_high": ci_high
    })

model_price_indices = pd.DataFrame(model_results).round(1)

highest_model_indices = (
    model_price_indices
    .sort_values(
        "conditional_price_index_portfolio_equals_100",
        ascending=False
    )
    .head(10)
)

lowest_model_indices = (
    model_price_indices
    .sort_values(
        "conditional_price_index_portfolio_equals_100",
        ascending=True
    )
    .head(10)
)


print("Manufacturer-level conditional price indices")
display(manufacturer_price_indices)

print("Ten highest supported named-model indices")
display(highest_model_indices)

print("Ten lowest supported named-model indices")
display(lowest_model_indices)

import numpy as np
import pandas as pd

# ---------------------------------------------------------
# Retrieve the untouched, complete-case test population
# ---------------------------------------------------------

diagnostic_test = pd.read_sql_query(
    """
    SELECT *
    FROM used_cars_regression_data
    WHERE data_split = 'test'
      AND engine_size_clean IS NOT NULL
      AND transmission_clean IS NOT NULL
    """,
    conn
).copy()


# Recreate A2 piecewise variables
diagnostic_test["age_after_1_year"] = np.maximum(
    diagnostic_test["age_years"] - 1, 0
)

diagnostic_test["age_after_3_years"] = np.maximum(
    diagnostic_test["age_years"] - 3, 0
)

diagnostic_test["age_after_5_years"] = np.maximum(
    diagnostic_test["age_years"] - 5, 0
)

diagnostic_test["age_after_10_years"] = np.maximum(
    diagnostic_test["age_years"] - 10, 0
)

diagnostic_test["mileage_10k"] = (
    diagnostic_test["mileage_miles"] / 10000
)

diagnostic_test["mileage_after_10k"] = np.maximum(
    diagnostic_test["mileage_10k"] - 1, 0
)

diagnostic_test["mileage_after_30k"] = np.maximum(
    diagnostic_test["mileage_10k"] - 3, 0
)

diagnostic_test["mileage_after_60k"] = np.maximum(
    diagnostic_test["mileage_10k"] - 6, 0
)

diagnostic_test["mileage_after_100k"] = np.maximum(
    diagnostic_test["mileage_10k"] - 10, 0
)


# ---------------------------------------------------------
# A2 held-out expected listed price
# ---------------------------------------------------------

a2_smearing_factor = np.mean(np.exp(model_a2.resid))

diagnostic_test["expected_listed_price_gbp"] = (
    np.exp(model_a2.predict(diagnostic_test))
    * a2_smearing_factor
)

diagnostic_test["expected_listed_price_gbp"] = (
    diagnostic_test["expected_listed_price_gbp"]
    .round(2)
)


# Positive = listed above model benchmark
# Negative = listed below model benchmark

diagnostic_test["price_position_gap_gbp"] = (
    diagnostic_test["listed_price_gbp"]
    - diagnostic_test["expected_listed_price_gbp"]
)

diagnostic_test["price_position_gap_pct"] = (
    100
    * diagnostic_test["price_position_gap_gbp"]
    / diagnostic_test["expected_listed_price_gbp"]
)


# ---------------------------------------------------------
# Held-out residual distribution
# ---------------------------------------------------------

percentiles = [
    0.00,
    0.01,
    0.025,
    0.05,
    0.25,
    0.50,
    0.75,
    0.95,
    0.975,
    0.99,
    1.00
]

position_distribution = (
    diagnostic_test["price_position_gap_pct"]
    .quantile(percentiles)
    .rename_axis("percentile")
    .reset_index(name="price_position_gap_pct")
)

position_distribution["percentile"] = (
    position_distribution["percentile"] * 100
)

position_distribution = position_distribution.round(2)


# ---------------------------------------------------------
# Illustrative tail-review lists
# ---------------------------------------------------------

lower_threshold = diagnostic_test[
    "price_position_gap_pct"
].quantile(0.025)

upper_threshold = diagnostic_test[
    "price_position_gap_pct"
].quantile(0.975)

diagnostic_test["positioning_diagnostic"] = np.select(
    [
        diagnostic_test["price_position_gap_pct"]
        <= lower_threshold,

        diagnostic_test["price_position_gap_pct"]
        >= upper_threshold
    ],
    [
        "Lower 2.5% versus model benchmark",
        "Upper 2.5% versus model benchmark"
    ],
    default="Central 95%"
)


output_columns = [
    "manufacturer",
    "model",
    "model_year",
    "approximate_vehicle_age",
    "mileage_miles",
    "listed_price_gbp",
    "expected_listed_price_gbp",
    "price_position_gap_gbp",
    "price_position_gap_pct",
    "positioning_diagnostic"
]

lowest_positioned_listings = (
    diagnostic_test
    .nsmallest(10, "price_position_gap_pct")
    [output_columns]
    .round(2)
)

highest_positioned_listings = (
    diagnostic_test
    .nlargest(10, "price_position_gap_pct")
    [output_columns]
    .round(2)
)


diagnostic_summary = pd.DataFrame({
    "test_records": [len(diagnostic_test)],
    "lower_2_5_percent_threshold": [lower_threshold],
    "upper_2_5_percent_threshold": [upper_threshold],
    "lower_tail_records": [
        (
            diagnostic_test["positioning_diagnostic"]
            == "Lower 2.5% versus model benchmark"
        ).sum()
    ],
    "upper_tail_records": [
        (
            diagnostic_test["positioning_diagnostic"]
            == "Upper 2.5% versus model benchmark"
        ).sum()
    ]
}).round(2)


print("Held-out price-position distribution")
display(position_distribution)

print("Diagnostic thresholds and record counts")
display(diagnostic_summary)

print("Listings furthest below the model benchmark")
display(lowest_positioned_listings)

print("Listings furthest above the model benchmark")
display(highest_positioned_listings)

import numpy as np
import pandas as pd

# ---------------------------------------------------------
# Training population for the strengthened comparable
# Test population remains the complete-case held-out set
# used for A2 validation.
# ---------------------------------------------------------

strong_comparable_train = pd.read_sql_query(
    """
    SELECT
        manufacturer,
        model,
        approximate_vehicle_age,
        mileage_miles,
        listed_price_gbp,
        non_price_fingerprint_id
    FROM used_cars_regression_data
    WHERE data_split = 'train'
    """,
    conn
).copy()

strong_comparable_test = diagnostic_test.copy()


def add_comparable_segments(data):
    data = data.copy()

    data["manufacturer_model"] = (
        data["manufacturer"].astype(str)
        + " | "
        + data["model"].astype(str)
    )

    data["age_band"] = pd.cut(
        data["approximate_vehicle_age"],
        bins=[-np.inf, 2, 5, 10, np.inf],
        labels=[
            "0-2 years",
            "3-5 years",
            "6-10 years",
            "11+ years"
        ]
    )

    data["mileage_band"] = pd.cut(
        data["mileage_miles"],
        bins=[
            -np.inf,
            4999,
            9999,
            19999,
            29999,
            39999,
            59999,
            79999,
            99999,
            149999,
            np.inf
        ],
        labels=[
            "00,000-04,999",
            "05,000-09,999",
            "10,000-19,999",
            "20,000-29,999",
            "30,000-39,999",
            "40,000-59,999",
            "60,000-79,999",
            "80,000-99,999",
            "100,000-149,999",
            "150,000+"
        ]
    )

    return data


strong_comparable_train = add_comparable_segments(
    strong_comparable_train
)

strong_comparable_test = add_comparable_segments(
    strong_comparable_test
)


# ---------------------------------------------------------
# Create training-only comparable lookup tables
# ---------------------------------------------------------

def create_comparable_lookup(data, keys, level_name):
    lookup = (
        data
        .groupby(keys, observed=True, dropna=False)
        .agg(
            comparable_median_price=(
                "listed_price_gbp",
                "median"
            ),
            fingerprint_support=(
                "non_price_fingerprint_id",
                "nunique"
            )
        )
        .reset_index()
    )

    lookup = lookup[
        lookup["fingerprint_support"] >= 25
    ].copy()

    lookup["comparable_level"] = level_name

    return lookup


comparable_levels = [
    (
        "01_model_age_mileage",
        ["manufacturer_model", "age_band", "mileage_band"]
    ),
    (
        "02_model_age",
        ["manufacturer_model", "age_band"]
    ),
    (
        "03_model",
        ["manufacturer_model"]
    ),
    (
        "04_manufacturer_age",
        ["manufacturer", "age_band"]
    ),
    (
        "05_manufacturer",
        ["manufacturer"]
    )
]

comparable_lookups = {
    level_name: create_comparable_lookup(
        strong_comparable_train,
        keys,
        level_name
    )
    for level_name, keys in comparable_levels
}


# ---------------------------------------------------------
# Apply the hierarchy sequentially to held-out cars
# ---------------------------------------------------------

strong_comparable_test[
    "strong_comparable_prediction"
] = np.nan

strong_comparable_test[
    "strong_comparable_level"
] = None

strong_comparable_test[
    "strong_comparable_support"
] = np.nan


for level_name, keys in comparable_levels:
    lookup = comparable_lookups[level_name]

    lookup_index = pd.MultiIndex.from_frame(
        lookup[keys]
    )

    price_lookup = pd.Series(
        lookup["comparable_median_price"].to_numpy(),
        index=lookup_index
    )

    support_lookup = pd.Series(
        lookup["fingerprint_support"].to_numpy(),
        index=lookup_index
    )

    test_index = pd.MultiIndex.from_frame(
        strong_comparable_test[keys]
    )

    candidate_prices = price_lookup.reindex(
        test_index
    ).to_numpy()

    candidate_support = support_lookup.reindex(
        test_index
    ).to_numpy()

    use_level = (
        strong_comparable_test[
            "strong_comparable_prediction"
        ].isna()
        & pd.notna(candidate_prices)
    )

    strong_comparable_test.loc[
        use_level,
        "strong_comparable_prediction"
    ] = candidate_prices[use_level]

    strong_comparable_test.loc[
        use_level,
        "strong_comparable_support"
    ] = candidate_support[use_level]

    strong_comparable_test.loc[
        use_level,
        "strong_comparable_level"
    ] = level_name


# Overall-median final fallback
overall_train_median = strong_comparable_train[
    "listed_price_gbp"
].median()

overall_fingerprint_support = strong_comparable_train[
    "non_price_fingerprint_id"
].nunique()

needs_overall = strong_comparable_test[
    "strong_comparable_prediction"
].isna()

strong_comparable_test.loc[
    needs_overall,
    "strong_comparable_prediction"
] = overall_train_median

strong_comparable_test.loc[
    needs_overall,
    "strong_comparable_support"
] = overall_fingerprint_support

strong_comparable_test.loc[
    needs_overall,
    "strong_comparable_level"
] = "06_overall"


# ---------------------------------------------------------
# Coverage by hierarchy level
# ---------------------------------------------------------

strong_comparable_coverage = (
    strong_comparable_test
    .groupby("strong_comparable_level", dropna=False)
    .agg(
        test_records=(
            "non_price_fingerprint_id",
            "size"
        ),
        fingerprint_groups=(
            "non_price_fingerprint_id",
            "nunique"
        ),
        minimum_fingerprint_support=(
            "strong_comparable_support",
            "min"
        ),
        average_fingerprint_support=(
            "strong_comparable_support",
            "mean"
        ),
        maximum_fingerprint_support=(
            "strong_comparable_support",
            "max"
        )
    )
    .reset_index()
)

strong_comparable_coverage[
    "test_record_percentage"
] = (
    100
    * strong_comparable_coverage["test_records"]
    / len(strong_comparable_test)
)

strong_comparable_coverage = (
    strong_comparable_coverage
    .sort_values("strong_comparable_level")
    .round(2)
)


# ---------------------------------------------------------
# Compare strengthened comparables against selected A2
# on exactly the same 19,433 held-out records
# ---------------------------------------------------------

def calculate_validation_metrics(
    actual,
    predicted,
    benchmark_name
):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    absolute_error = np.abs(predicted - actual)

    absolute_percentage_error = (
        100 * absolute_error / actual
    )

    signed_percentage_error = (
        100 * (predicted - actual) / actual
    )

    return {
        "benchmark": benchmark_name,
        "test_records": len(actual),
        "mae_gbp": absolute_error.mean(),
        "median_absolute_error_gbp":
            np.median(absolute_error),
        "mdape_percentage":
            np.median(absolute_percentage_error),
        "within_10_percent":
            100 * np.mean(
                absolute_percentage_error <= 10
            ),
        "within_15_percent":
            100 * np.mean(
                absolute_percentage_error <= 15
            ),
        "within_20_percent":
            100 * np.mean(
                absolute_percentage_error <= 20
            ),
        "median_signed_percentage_error":
            np.median(signed_percentage_error)
    }


strong_baseline_metrics = pd.DataFrame([
    calculate_validation_metrics(
        strong_comparable_test["listed_price_gbp"],
        strong_comparable_test[
            "strong_comparable_prediction"
        ],
        "Strengthened hierarchical comparables"
    ),
    calculate_validation_metrics(
        strong_comparable_test["listed_price_gbp"],
        strong_comparable_test[
            "expected_listed_price_gbp"
        ],
        "Model A2: piecewise log price"
    )
]).round(2)


print("Strengthened comparable coverage")
display(strong_comparable_coverage)

print("Strengthened baseline versus Model A2")
display(strong_baseline_metrics)

import pandas as pd

# Standard-guidance ages only
selected_ages = [0, 1, 3, 5, 7, 10]

# Selected mileage reference points and the observed mileage
# bands used to assess support around each point
selected_mileage_points = {
    0: "00,000-04,999",
    10000: "10,000-19,999",
    30000: "30,000-39,999",
    60000: "60,000-79,999",
    100000: "100,000-149,999"
}


# ---------------------------------------------------------
# Actual training fingerprint support by exact age and
# corresponding mileage band
# ---------------------------------------------------------

joint_support = (
    strong_comparable_train[
        strong_comparable_train[
            "approximate_vehicle_age"
        ].isin(selected_ages)
        &
        strong_comparable_train[
            "mileage_band"
        ].astype(str).isin(
            selected_mileage_points.values()
        )
    ]
    .groupby(
        ["approximate_vehicle_age", "mileage_band"],
        observed=True
    )
    ["non_price_fingerprint_id"]
    .nunique()
    .rename("training_fingerprint_support")
)


# ---------------------------------------------------------
# A2-implied joint conditional price index
# Reference: age 0 and mileage 0 = 100
# ---------------------------------------------------------

joint_results = []

for age in selected_ages:
    for mileage, support_band in (
        selected_mileage_points.items()
    ):
        mileage_10k = mileage / 10000

        joint_terms = {
            # Age terms
            "age_years": age,
            "age_after_1_year": max(age - 1, 0),
            "age_after_3_years": max(age - 3, 0),
            "age_after_5_years": max(age - 5, 0),
            "age_after_10_years": max(age - 10, 0),

            # Mileage terms
            "mileage_10k": mileage_10k,
            "mileage_after_10k":
                max(mileage_10k - 1, 0),
            "mileage_after_30k":
                max(mileage_10k - 3, 0),
            "mileage_after_60k":
                max(mileage_10k - 6, 0),
            "mileage_after_100k":
                max(mileage_10k - 10, 0)
        }

        price_index, ci_low, ci_high = (
            calculate_index_and_ci(joint_terms)
        )

        support = joint_support.get(
            (age, support_band),
            0
        )

        if support >= 100:
            support_status = "Strong support"
        elif support >= 25:
            support_status = "Adequate support"
        else:
            support_status = "Low support / directional only"

        joint_results.append({
            "approximate_vehicle_age": age,
            "model_year": 2020 - age,
            "mileage_reference_miles": mileage,
            "mileage_support_band": support_band,
            "joint_adjusted_price_index": price_index,
            "confidence_interval_low": ci_low,
            "confidence_interval_high": ci_high,
            "training_fingerprint_support": support,
            "support_status": support_status
        })


joint_age_mileage_schedule = (
    pd.DataFrame(joint_results)
    .round({
        "joint_adjusted_price_index": 1,
        "confidence_interval_low": 1,
        "confidence_interval_high": 1
    })
)


# Compact index matrix for the eventual presentation
joint_price_index_matrix = (
    joint_age_mileage_schedule
    .pivot(
        index="approximate_vehicle_age",
        columns="mileage_reference_miles",
        values="joint_adjusted_price_index"
    )
    .round(1)
)

joint_price_index_matrix.columns = [
    f"{int(mileage):,} miles"
    for mileage in joint_price_index_matrix.columns
]


# Matching support matrix
joint_support_matrix = (
    joint_age_mileage_schedule
    .pivot(
        index="approximate_vehicle_age",
        columns="mileage_reference_miles",
        values="training_fingerprint_support"
    )
)

joint_support_matrix.columns = [
    f"{int(mileage):,} miles"
    for mileage in joint_support_matrix.columns
]


print(
    "Joint adjusted price index: "
    "age 0 and mileage 0 = 100"
)
display(joint_price_index_matrix)

print("Training fingerprint support around each combination")
display(joint_support_matrix)

print("Detailed joint schedule with confidence and support")
display(joint_age_mileage_schedule)

import numpy as np
import pandas as pd

qa_results = []


def record_check(check_name, condition, observed):
    qa_results.append({
        "check": check_name,
        "status": "PASS" if bool(condition) else "FAIL",
        "observed": str(observed)
    })


# ---------------------------------------------------------
# 1. Split and complete-case reconciliation
# ---------------------------------------------------------

population_audit = pd.read_sql_query(
    """
    SELECT
        data_split,
        COUNT(*) AS total_records,
        COUNT(
            DISTINCT non_price_fingerprint_id
        ) AS total_fingerprints,

        SUM(
            CASE
                WHEN engine_size_clean IS NOT NULL
                 AND transmission_clean IS NOT NULL
                THEN 1
                ELSE 0
            END
        ) AS complete_case_records,

        COUNT(
            DISTINCT CASE
                WHEN engine_size_clean IS NOT NULL
                 AND transmission_clean IS NOT NULL
                THEN non_price_fingerprint_id
            END
        ) AS complete_case_fingerprints

    FROM used_cars_regression_data
    GROUP BY data_split
    ORDER BY data_split
    """,
    conn
)

population_lookup = (
    population_audit
    .set_index("data_split")
    .to_dict("index")
)

expected_population = {
    "train": {
        "total_records": 78173,
        "total_fingerprints": 76755,
        "complete_case_records": 77965,
        "complete_case_fingerprints": 76555
    },
    "test": {
        "total_records": 19500,
        "total_fingerprints": 19190,
        "complete_case_records": 19433,
        "complete_case_fingerprints": 19125
    }
}

for split_name, expected_values in expected_population.items():
    for field_name, expected_value in expected_values.items():
        observed_value = int(
            population_lookup[split_name][field_name]
        )

        record_check(
            f"{split_name}: {field_name}",
            observed_value == expected_value,
            observed_value
        )


# ---------------------------------------------------------
# 2. Confirm zero fingerprint leakage
# ---------------------------------------------------------

overlap_count = pd.read_sql_query(
    """
    SELECT COUNT(*) AS overlapping_fingerprints
    FROM (
        SELECT non_price_fingerprint_id
        FROM used_cars_regression_data
        GROUP BY non_price_fingerprint_id
        HAVING COUNT(DISTINCT data_split) > 1
    ) overlap_check
    """,
    conn
).iloc[0]["overlapping_fingerprints"]

record_check(
    "Zero fingerprints shared across train and test",
    int(overlap_count) == 0,
    int(overlap_count)
)


# ---------------------------------------------------------
# 3. Confirm selected A2 specification
# ---------------------------------------------------------

required_a2_terms = {
    "age_years",
    "age_after_1_year",
    "age_after_3_years",
    "age_after_5_years",
    "age_after_10_years",
    "mileage_10k",
    "mileage_after_10k",
    "mileage_after_30k",
    "mileage_after_60k",
    "mileage_after_100k",
    "engine_size_clean"
}

missing_a2_terms = (
    required_a2_terms
    - set(model_a2.params.index)
)

record_check(
    "All required A2 terms present",
    len(missing_a2_terms) == 0,
    sorted(missing_a2_terms)
)

record_check(
    "A2 training records",
    int(model_a2.nobs) == 77965,
    int(model_a2.nobs)
)

model_frame = model_a2.model.data.frame

record_check(
    "A2 complete-case training fingerprints",
    model_frame[
        "non_price_fingerprint_id"
    ].nunique() == 76555,
    model_frame[
        "non_price_fingerprint_id"
    ].nunique()
)

record_check(
    "A2 regression model groups",
    model_frame[
        "regression_model_group"
    ].nunique() == 147,
    model_frame[
        "regression_model_group"
    ].nunique()
)

record_check(
    "A2 adjusted R-squared",
    np.isclose(
        model_a2.rsquared_adj,
        0.9441,
        atol=0.0001
    ),
    round(model_a2.rsquared_adj, 4)
)

qa_smearing_factor = np.mean(
    np.exp(model_a2.resid)
)

record_check(
    "A2 smearing factor",
    np.isclose(
        qa_smearing_factor,
        1.0081,
        atol=0.0001
    ),
    round(qa_smearing_factor, 4)
)


# No separately supported electric effect
electric_terms = [
    term
    for term in model_a2.params.index
    if "Electric" in term
]

record_check(
    "No unsupported standalone Electric coefficient",
    len(electric_terms) == 0,
    electric_terms
)


# ---------------------------------------------------------
# 4. Recompute A2 and strengthened-comparable metrics
# ---------------------------------------------------------

record_check(
    "Diagnostic test records",
    len(diagnostic_test) == 19433,
    len(diagnostic_test)
)

record_check(
    "Diagnostic test fingerprints",
    diagnostic_test[
        "non_price_fingerprint_id"
    ].nunique() == 19125,
    diagnostic_test[
        "non_price_fingerprint_id"
    ].nunique()
)

record_check(
    "No missing A2 predictions",
    diagnostic_test[
        "expected_listed_price_gbp"
    ].notna().all(),
    diagnostic_test[
        "expected_listed_price_gbp"
    ].isna().sum()
)

record_check(
    "All A2 predictions are positive",
    (
        diagnostic_test[
            "expected_listed_price_gbp"
        ] > 0
    ).all(),
    diagnostic_test[
        "expected_listed_price_gbp"
    ].min()
)

record_check(
    "No missing strengthened-comparable predictions",
    strong_comparable_test[
        "strong_comparable_prediction"
    ].notna().all(),
    strong_comparable_test[
        "strong_comparable_prediction"
    ].isna().sum()
)


def recompute_metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    absolute_error = np.abs(predicted - actual)

    absolute_percentage_error = (
        100 * absolute_error / actual
    )

    signed_percentage_error = (
        100 * (predicted - actual) / actual
    )

    return {
        "mae": absolute_error.mean(),
        "median_ae": np.median(absolute_error),
        "mdape": np.median(
            absolute_percentage_error
        ),
        "within_10": 100 * np.mean(
            absolute_percentage_error <= 10
        ),
        "within_15": 100 * np.mean(
            absolute_percentage_error <= 15
        ),
        "within_20": 100 * np.mean(
            absolute_percentage_error <= 20
        ),
        "median_bias": np.median(
            signed_percentage_error
        )
    }


a2_qa_metrics = recompute_metrics(
    diagnostic_test["listed_price_gbp"],
    diagnostic_test[
        "expected_listed_price_gbp"
    ]
)

comparable_qa_metrics = recompute_metrics(
    strong_comparable_test["listed_price_gbp"],
    strong_comparable_test[
        "strong_comparable_prediction"
    ]
)


expected_a2_metrics = {
    "mae": 1613.11,
    "median_ae": 1057.47,
    "mdape": 7.77,
    "within_10": 61.28,
    "within_15": 80.04,
    "within_20": 90.35,
    "median_bias": 1.03
}

expected_comparable_metrics = {
    "mae": 2355.01,
    "median_ae": 1335.00,
    "mdape": 9.61,
    "within_10": 51.38,
    "within_15": 67.65,
    "within_20": 78.60,
    "median_bias": 0.07
}


for metric_name, expected_value in (
    expected_a2_metrics.items()
):
    observed_value = a2_qa_metrics[metric_name]

    record_check(
        f"A2 metric: {metric_name}",
        np.isclose(
            observed_value,
            expected_value,
            atol=0.02
        ),
        round(observed_value, 2)
    )


for metric_name, expected_value in (
    expected_comparable_metrics.items()
):
    observed_value = comparable_qa_metrics[
        metric_name
    ]

    record_check(
        f"Comparable metric: {metric_name}",
        np.isclose(
            observed_value,
            expected_value,
            atol=0.02
        ),
        round(observed_value, 2)
    )


# ---------------------------------------------------------
# 5. Confirm strengthened hierarchy reconciliation
# ---------------------------------------------------------

expected_level_counts = {
    "01_model_age_mileage": 16517,
    "02_model_age": 2486,
    "03_model": 330,
    "04_manufacturer_age": 99,
    "05_manufacturer": 1
}

observed_level_counts = (
    strong_comparable_test[
        "strong_comparable_level"
    ]
    .value_counts()
    .to_dict()
)

for level_name, expected_count in (
    expected_level_counts.items()
):
    observed_count = observed_level_counts.get(
        level_name,
        0
    )

    record_check(
        f"Comparable coverage: {level_name}",
        observed_count == expected_count,
        observed_count
    )

record_check(
    "Comparable hierarchy covers all held-out records",
    sum(observed_level_counts.values()) == 19433,
    sum(observed_level_counts.values())
)


# ---------------------------------------------------------
# Final QA output
# ---------------------------------------------------------

step_4_qa = pd.DataFrame(qa_results)

display(step_4_qa)

failed_checks = step_4_qa[
    step_4_qa["status"] == "FAIL"
]

if len(failed_checks) == 0:
    print(
        "STEP 4 REPRODUCIBILITY CHECK: "
        "ALL CHECKS PASSED"
    )
else:
    raise AssertionError(
        "Step 4 is not reproducible yet. "
        f"{len(failed_checks)} checks failed."
    )

import numpy as np
import pandas as pd

interval_data = diagnostic_test.copy()

# ---------------------------------------------------------
# Fingerprint-safe deterministic division of the existing
# held-out population
# ---------------------------------------------------------

fingerprint_hash = pd.util.hash_pandas_object(
    interval_data[
        "non_price_fingerprint_id"
    ].astype(str),
    index=False
).astype("uint64")

interval_data["interval_sample"] = np.where(
    fingerprint_hash % 2 == 0,
    "calibration",
    "validation"
)

calibration_data = interval_data[
    interval_data["interval_sample"]
    == "calibration"
].copy()

interval_validation_data = interval_data[
    interval_data["interval_sample"]
    == "validation"
].copy()


# Confirm fingerprint independence
calibration_fingerprints = set(
    calibration_data[
        "non_price_fingerprint_id"
    ]
)

validation_fingerprints = set(
    interval_validation_data[
        "non_price_fingerprint_id"
    ]
)

interval_fingerprint_overlap = len(
    calibration_fingerprints
    & validation_fingerprints
)


# ---------------------------------------------------------
# Observed-price-to-benchmark ratio
# ---------------------------------------------------------

calibration_data["actual_to_expected_ratio"] = (
    calibration_data["listed_price_gbp"]
    / calibration_data["expected_listed_price_gbp"]
)


# Empirical multiplicative limits
interval_limits = {
    "90% empirical range": {
        "lower_multiplier": calibration_data[
            "actual_to_expected_ratio"
        ].quantile(0.05),
        "upper_multiplier": calibration_data[
            "actual_to_expected_ratio"
        ].quantile(0.95)
    },
    "95% empirical range": {
        "lower_multiplier": calibration_data[
            "actual_to_expected_ratio"
        ].quantile(0.025),
        "upper_multiplier": calibration_data[
            "actual_to_expected_ratio"
        ].quantile(0.975)
    }
}


# ---------------------------------------------------------
# Apply calibration limits to the separate validation half
# ---------------------------------------------------------

for range_name, limits in interval_limits.items():
    short_name = (
        "90"
        if range_name.startswith("90")
        else "95"
    )

    interval_validation_data[
        f"lower_{short_name}_price"
    ] = (
        interval_validation_data[
            "expected_listed_price_gbp"
        ]
        * limits["lower_multiplier"]
    )

    interval_validation_data[
        f"upper_{short_name}_price"
    ] = (
        interval_validation_data[
            "expected_listed_price_gbp"
        ]
        * limits["upper_multiplier"]
    )

    interval_validation_data[
        f"inside_{short_name}_range"
    ] = (
        interval_validation_data[
            "listed_price_gbp"
        ].between(
            interval_validation_data[
                f"lower_{short_name}_price"
            ],
            interval_validation_data[
                f"upper_{short_name}_price"
            ],
            inclusive="both"
        )
    )


# ---------------------------------------------------------
# Overall independently measured coverage
# ---------------------------------------------------------

interval_summary = []

for range_name, limits in interval_limits.items():
    short_name = (
        "90"
        if range_name.startswith("90")
        else "95"
    )

    interval_summary.append({
        "empirical_range": range_name,
        "calibration_records":
            len(calibration_data),
        "calibration_fingerprints":
            calibration_data[
                "non_price_fingerprint_id"
            ].nunique(),
        "validation_records":
            len(interval_validation_data),
        "validation_fingerprints":
            interval_validation_data[
                "non_price_fingerprint_id"
            ].nunique(),
        "fingerprints_in_both_samples":
            interval_fingerprint_overlap,
        "lower_price_multiplier":
            limits["lower_multiplier"],
        "upper_price_multiplier":
            limits["upper_multiplier"],
        "validation_coverage_percentage":
            100
            * interval_validation_data[
                f"inside_{short_name}_range"
            ].mean(),
        "example_lower_price_for_15000":
            15000 * limits["lower_multiplier"],
        "example_upper_price_for_15000":
            15000 * limits["upper_multiplier"]
    })

interval_coverage_summary = (
    pd.DataFrame(interval_summary)
    .round(2)
)


# ---------------------------------------------------------
# Segment coverage
# ---------------------------------------------------------

interval_validation_data["age_band"] = pd.cut(
    interval_validation_data[
        "approximate_vehicle_age"
    ],
    bins=[-np.inf, 2, 5, 10, np.inf],
    labels=[
        "0-2 years",
        "3-5 years",
        "6-10 years",
        "11+ years"
    ]
)

interval_validation_data["mileage_band"] = pd.cut(
    interval_validation_data[
        "mileage_miles"
    ],
    bins=[
        -np.inf,
        9999,
        29999,
        59999,
        99999,
        np.inf
    ],
    labels=[
        "Under 10k",
        "10k-29,999",
        "30k-59,999",
        "60k-99,999",
        "100k+"
    ]
)

interval_validation_data["price_band"] = pd.cut(
    interval_validation_data[
        "listed_price_gbp"
    ],
    bins=[
        -np.inf,
        9999,
        19999,
        29999,
        49999,
        np.inf
    ],
    labels=[
        "Under £10k",
        "£10k-£20k",
        "£20k-£30k",
        "£30k-£50k",
        "£50k+"
    ]
)


segment_tables = []

segment_dimensions = [
    ("Manufacturer", "manufacturer"),
    ("Age band", "age_band"),
    ("Mileage band", "mileage_band"),
    ("Price band", "price_band")
]

for dimension_name, field_name in segment_dimensions:
    segment_result = (
        interval_validation_data
        .groupby(
            field_name,
            observed=True,
            dropna=False
        )
        .agg(
            validation_records=(
                "listed_price_gbp",
                "size"
            ),
            coverage_90_percentage=(
                "inside_90_range",
                lambda values:
                    100 * values.mean()
            ),
            coverage_95_percentage=(
                "inside_95_range",
                lambda values:
                    100 * values.mean()
            )
        )
        .reset_index()
        .rename(columns={
            field_name: "segment"
        })
    )

    segment_result.insert(
        0,
        "dimension",
        dimension_name
    )

    segment_tables.append(segment_result)

interval_segment_coverage = (
    pd.concat(
        segment_tables,
        ignore_index=True
    )
    .round(2)
)


print("Empirical price-range calibration and validation")
display(interval_coverage_summary)

print("Empirical price-range coverage by segment")
display(interval_segment_coverage)

import numpy as np
import pandas as pd

# ---------------------------------------------------------
# Correct locked analytical population
# ---------------------------------------------------------

tier_population = pd.read_sql_query(
    """
    SELECT *
    FROM used_cars_regression_data
    """,
    conn
).copy()

assert len(tier_population) == 97673


# Add the same comparable segments used in the final baseline
tier_population = add_comparable_segments(
    tier_population
)


# ---------------------------------------------------------
# Generate A2 benchmarks where required inputs are complete
# ---------------------------------------------------------

tier_population["age_after_1_year"] = np.maximum(
    tier_population["age_years"] - 1,
    0
)

tier_population["age_after_3_years"] = np.maximum(
    tier_population["age_years"] - 3,
    0
)

tier_population["age_after_5_years"] = np.maximum(
    tier_population["age_years"] - 5,
    0
)

tier_population["age_after_10_years"] = np.maximum(
    tier_population["age_years"] - 10,
    0
)

tier_population["mileage_10k"] = (
    tier_population["mileage_miles"] / 10000
)

tier_population["mileage_after_10k"] = np.maximum(
    tier_population["mileage_10k"] - 1,
    0
)

tier_population["mileage_after_30k"] = np.maximum(
    tier_population["mileage_10k"] - 3,
    0
)

tier_population["mileage_after_60k"] = np.maximum(
    tier_population["mileage_10k"] - 6,
    0
)

tier_population["mileage_after_100k"] = np.maximum(
    tier_population["mileage_10k"] - 10,
    0
)

complete_for_a2 = (
    tier_population["engine_size_clean"].notna()
    & tier_population["transmission_clean"].notna()
)

tier_population["expected_listed_price_gbp"] = np.nan

tier_population.loc[
    complete_for_a2,
    "expected_listed_price_gbp"
] = (
    np.exp(
        model_a2.predict(
            tier_population.loc[complete_for_a2]
        )
    )
    * qa_smearing_factor
)


# ---------------------------------------------------------
# Apply the strengthened training-only comparable hierarchy
# to the whole analytical portfolio
# ---------------------------------------------------------

tier_population["comparable_level"] = None
tier_population["comparable_support"] = np.nan
tier_population["comparable_median_price"] = np.nan

for level_name, keys in comparable_levels:
    lookup = comparable_lookups[level_name]

    lookup_index = pd.MultiIndex.from_frame(
        lookup[keys]
    )

    price_lookup = pd.Series(
        lookup["comparable_median_price"].to_numpy(),
        index=lookup_index
    )

    support_lookup = pd.Series(
        lookup["fingerprint_support"].to_numpy(),
        index=lookup_index
    )

    population_index = pd.MultiIndex.from_frame(
        tier_population[keys]
    )

    candidate_price = price_lookup.reindex(
        population_index
    ).to_numpy()

    candidate_support = support_lookup.reindex(
        population_index
    ).to_numpy()

    use_level = (
        tier_population[
            "comparable_median_price"
        ].isna()
        & pd.notna(candidate_price)
    )

    tier_population.loc[
        use_level,
        "comparable_median_price"
    ] = candidate_price[use_level]

    tier_population.loc[
        use_level,
        "comparable_support"
    ] = candidate_support[use_level]

    tier_population.loc[
        use_level,
        "comparable_level"
    ] = level_name


# Final overall-median fallback
needs_overall = tier_population[
    "comparable_median_price"
].isna()

tier_population.loc[
    needs_overall,
    "comparable_median_price"
] = overall_train_median

tier_population.loc[
    needs_overall,
    "comparable_support"
] = overall_fingerprint_support

tier_population.loc[
    needs_overall,
    "comparable_level"
] = "06_overall"


# ---------------------------------------------------------
# Routing rules without an unvalidated price trigger
# ---------------------------------------------------------

pooled_model = (
    tier_population["regression_model_group"]
    .fillna("")
    .str.endswith("Other models")
)

tier_3_condition = (
    (tier_population["approximate_vehicle_age"] >= 11)
    | (tier_population["mileage_miles"] >= 100000)
    | (
        tier_population["fuel_type_analysis"]
        == "Other / unclear"
    )
    | tier_population["engine_size_clean"].isna()
    | tier_population["transmission_clean"].isna()
    | pooled_model
    | tier_population[
        "expected_listed_price_gbp"
    ].isna()
)

tier_2_condition = (
    ~tier_3_condition
    & (
        tier_population[
            "approximate_vehicle_age"
        ].between(6, 10)
        | tier_population[
            "mileage_miles"
        ].between(60000, 99999)
        | (
            tier_population["comparable_level"]
            != "01_model_age_mileage"
        )
    )
)

tier_population["guidance_tier"] = np.select(
    [
        tier_3_condition,
        tier_2_condition
    ],
    [
        "Tier 3 — Manual appraisal",
        "Tier 2 — Cautious guidance"
    ],
    default="Tier 1 — Standard guidance"
)


# ---------------------------------------------------------
# Correct tier sizing
# ---------------------------------------------------------

total_listed_value = tier_population[
    "listed_price_gbp"
].sum()

tier_summary = (
    tier_population
    .groupby("guidance_tier")
    .agg(
        listings=(
            "listed_price_gbp",
            "size"
        ),
        aggregate_listed_price_value=(
            "listed_price_gbp",
            "sum"
        ),
        median_listed_price=(
            "listed_price_gbp",
            "median"
        ),
        median_expected_price=(
            "expected_listed_price_gbp",
            "median"
        )
    )
    .reset_index()
)

tier_summary["listing_share_percentage"] = (
    100
    * tier_summary["listings"]
    / len(tier_population)
)

tier_summary[
    "aggregate_listed_price_share_percentage"
] = (
    100
    * tier_summary[
        "aggregate_listed_price_value"
    ]
    / total_listed_value
)

tier_summary = tier_summary[
    [
        "guidance_tier",
        "listings",
        "listing_share_percentage",
        "aggregate_listed_price_value",
        "aggregate_listed_price_share_percentage",
        "median_listed_price",
        "median_expected_price"
    ]
].round(2)


tier_reconciliation = pd.DataFrame({
    "analytical_population": [len(tier_population)],
    "tiered_records": [tier_summary["listings"].sum()],
    "difference": [
        len(tier_population)
        - tier_summary["listings"].sum()
    ]
})


# ---------------------------------------------------------
# Validate proposed predicted-price triggers using the
# fingerprint-separated interval-validation population
# ---------------------------------------------------------

predicted_price_validation = (
    interval_validation_data.copy()
)

predicted_price_validation[
    "predicted_price_band"
] = pd.cut(
    predicted_price_validation[
        "expected_listed_price_gbp"
    ],
    bins=[
        -np.inf,
        29999.99,
        49999.99,
        np.inf
    ],
    labels=[
        "Predicted below £30k",
        "Predicted £30k-£49,999",
        "Predicted £50k+"
    ]
)

predicted_price_validation[
    "absolute_percentage_error"
] = (
    100
    * (
        predicted_price_validation[
            "expected_listed_price_gbp"
        ]
        - predicted_price_validation[
            "listed_price_gbp"
        ]
    ).abs()
    / predicted_price_validation[
        "listed_price_gbp"
    ]
)

predicted_price_validation[
    "signed_percentage_error"
] = (
    100
    * (
        predicted_price_validation[
            "expected_listed_price_gbp"
        ]
        - predicted_price_validation[
            "listed_price_gbp"
        ]
    )
    / predicted_price_validation[
        "listed_price_gbp"
    ]
)

predicted_price_band_validation = (
    predicted_price_validation
    .groupby(
        "predicted_price_band",
        observed=True
    )
    .agg(
        validation_records=(
            "listed_price_gbp",
            "size"
        ),
        median_actual_price=(
            "listed_price_gbp",
            "median"
        ),
        median_predicted_price=(
            "expected_listed_price_gbp",
            "median"
        ),
        mdape_percentage=(
            "absolute_percentage_error",
            "median"
        ),
        within_20_percent=(
            "absolute_percentage_error",
            lambda values:
                100 * (values <= 20).mean()
        ),
        median_signed_error=(
            "signed_percentage_error",
            "median"
        ),
        empirical_90_range_coverage=(
            "inside_90_range",
            lambda values:
                100 * values.mean()
        ),
        empirical_95_range_coverage=(
            "inside_95_range",
            lambda values:
                100 * values.mean()
        )
    )
    .reset_index()
    .round(2)
)


print("Guidance tier sizing on the locked population")
display(tier_summary)

print("Tier reconciliation")
display(tier_reconciliation)

print("Validation by predicted-price band")
display(predicted_price_band_validation)

import numpy as np
import pandas as pd

# ---------------------------------------------------------
# Correct locked analytical population
# ---------------------------------------------------------

tier_population = pd.read_sql_query(
    """
    SELECT *
    FROM used_cars_regression_data
    """,
    conn
).copy()

assert len(tier_population) == 97673


# Add the same comparable segments used in the final baseline
tier_population = add_comparable_segments(
    tier_population
)


# ---------------------------------------------------------
# Generate A2 benchmarks where required inputs are complete
# ---------------------------------------------------------

tier_population["age_after_1_year"] = np.maximum(
    tier_population["age_years"] - 1,
    0
)

tier_population["age_after_3_years"] = np.maximum(
    tier_population["age_years"] - 3,
    0
)

tier_population["age_after_5_years"] = np.maximum(
    tier_population["age_years"] - 5,
    0
)

tier_population["age_after_10_years"] = np.maximum(
    tier_population["age_years"] - 10,
    0
)

tier_population["mileage_10k"] = (
    tier_population["mileage_miles"] / 10000
)

tier_population["mileage_after_10k"] = np.maximum(
    tier_population["mileage_10k"] - 1,
    0
)

tier_population["mileage_after_30k"] = np.maximum(
    tier_population["mileage_10k"] - 3,
    0
)

tier_population["mileage_after_60k"] = np.maximum(
    tier_population["mileage_10k"] - 6,
    0
)

tier_population["mileage_after_100k"] = np.maximum(
    tier_population["mileage_10k"] - 10,
    0
)

complete_for_a2 = (
    tier_population["engine_size_clean"].notna()
    & tier_population["transmission_clean"].notna()
)

tier_population["expected_listed_price_gbp"] = np.nan

tier_population.loc[
    complete_for_a2,
    "expected_listed_price_gbp"
] = (
    np.exp(
        model_a2.predict(
            tier_population.loc[complete_for_a2]
        )
    )
    * qa_smearing_factor
)


# ---------------------------------------------------------
# Apply the strengthened training-only comparable hierarchy
# to the whole analytical portfolio
# ---------------------------------------------------------

tier_population["comparable_level"] = None
tier_population["comparable_support"] = np.nan
tier_population["comparable_median_price"] = np.nan

for level_name, keys in comparable_levels:
    lookup = comparable_lookups[level_name]

    lookup_index = pd.MultiIndex.from_frame(
        lookup[keys]
    )

    price_lookup = pd.Series(
        lookup["comparable_median_price"].to_numpy(),
        index=lookup_index
    )

    support_lookup = pd.Series(
        lookup["fingerprint_support"].to_numpy(),
        index=lookup_index
    )

    population_index = pd.MultiIndex.from_frame(
        tier_population[keys]
    )

    candidate_price = price_lookup.reindex(
        population_index
    ).to_numpy()

    candidate_support = support_lookup.reindex(
        population_index
    ).to_numpy()

    use_level = (
        tier_population[
            "comparable_median_price"
        ].isna()
        & pd.notna(candidate_price)
    )

    tier_population.loc[
        use_level,
        "comparable_median_price"
    ] = candidate_price[use_level]

    tier_population.loc[
        use_level,
        "comparable_support"
    ] = candidate_support[use_level]

    tier_population.loc[
        use_level,
        "comparable_level"
    ] = level_name


# Final overall-median fallback
needs_overall = tier_population[
    "comparable_median_price"
].isna()

tier_population.loc[
    needs_overall,
    "comparable_median_price"
] = overall_train_median

tier_population.loc[
    needs_overall,
    "comparable_support"
] = overall_fingerprint_support

tier_population.loc[
    needs_overall,
    "comparable_level"
] = "06_overall"


# ---------------------------------------------------------
# Routing rules without an unvalidated price trigger
# ---------------------------------------------------------

pooled_model = (
    tier_population["regression_model_group"]
    .fillna("")
    .str.endswith("Other models")
)

tier_3_condition = (
    (tier_population["approximate_vehicle_age"] >= 11)
    | (tier_population["mileage_miles"] >= 100000)
    | (
        tier_population["fuel_type_analysis"]
        == "Other / unclear"
    )
    | tier_population["engine_size_clean"].isna()
    | tier_population["transmission_clean"].isna()
    | pooled_model
    | tier_population[
        "expected_listed_price_gbp"
    ].isna()
)

tier_2_condition = (
    ~tier_3_condition
    & (
        tier_population[
            "approximate_vehicle_age"
        ].between(6, 10)
        | tier_population[
            "mileage_miles"
        ].between(60000, 99999)
        | (
            tier_population["comparable_level"]
            != "01_model_age_mileage"
        )
    )
)

tier_population["guidance_tier"] = np.select(
    [
        tier_3_condition,
        tier_2_condition
    ],
    [
        "Tier 3 — Manual appraisal",
        "Tier 2 — Cautious guidance"
    ],
    default="Tier 1 — Standard guidance"
)


# ---------------------------------------------------------
# Correct tier sizing
# ---------------------------------------------------------

total_listed_value = tier_population[
    "listed_price_gbp"
].sum()

tier_summary = (
    tier_population
    .groupby("guidance_tier")
    .agg(
        listings=(
            "listed_price_gbp",
            "size"
        ),
        aggregate_listed_price_value=(
            "listed_price_gbp",
            "sum"
        ),
        median_listed_price=(
            "listed_price_gbp",
            "median"
        ),
        median_expected_price=(
            "expected_listed_price_gbp",
            "median"
        )
    )
    .reset_index()
)

tier_summary["listing_share_percentage"] = (
    100
    * tier_summary["listings"]
    / len(tier_population)
)

tier_summary[
    "aggregate_listed_price_share_percentage"
] = (
    100
    * tier_summary[
        "aggregate_listed_price_value"
    ]
    / total_listed_value
)

tier_summary = tier_summary[
    [
        "guidance_tier",
        "listings",
        "listing_share_percentage",
        "aggregate_listed_price_value",
        "aggregate_listed_price_share_percentage",
        "median_listed_price",
        "median_expected_price"
    ]
].round(2)


tier_reconciliation = pd.DataFrame({
    "analytical_population": [len(tier_population)],
    "tiered_records": [tier_summary["listings"].sum()],
    "difference": [
        len(tier_population)
        - tier_summary["listings"].sum()
    ]
})


# ---------------------------------------------------------
# Validate proposed predicted-price triggers using the
# fingerprint-separated interval-validation population
# ---------------------------------------------------------

predicted_price_validation = (
    interval_validation_data.copy()
)

predicted_price_validation[
    "predicted_price_band"
] = pd.cut(
    predicted_price_validation[
        "expected_listed_price_gbp"
    ],
    bins=[
        -np.inf,
        29999.99,
        49999.99,
        np.inf
    ],
    labels=[
        "Predicted below £30k",
        "Predicted £30k-£49,999",
        "Predicted £50k+"
    ]
)

predicted_price_validation[
    "absolute_percentage_error"
] = (
    100
    * (
        predicted_price_validation[
            "expected_listed_price_gbp"
        ]
        - predicted_price_validation[
            "listed_price_gbp"
        ]
    ).abs()
    / predicted_price_validation[
        "listed_price_gbp"
    ]
)

predicted_price_validation[
    "signed_percentage_error"
] = (
    100
    * (
        predicted_price_validation[
            "expected_listed_price_gbp"
        ]
        - predicted_price_validation[
            "listed_price_gbp"
        ]
    )
    / predicted_price_validation[
        "listed_price_gbp"
    ]
)

predicted_price_band_validation = (
    predicted_price_validation
    .groupby(
        "predicted_price_band",
        observed=True
    )
    .agg(
        validation_records=(
            "listed_price_gbp",
            "size"
        ),
        median_actual_price=(
            "listed_price_gbp",
            "median"
        ),
        median_predicted_price=(
            "expected_listed_price_gbp",
            "median"
        ),
        mdape_percentage=(
            "absolute_percentage_error",
            "median"
        ),
        within_20_percent=(
            "absolute_percentage_error",
            lambda values:
                100 * (values <= 20).mean()
        ),
        median_signed_error=(
            "signed_percentage_error",
            "median"
        ),
        empirical_90_range_coverage=(
            "inside_90_range",
            lambda values:
                100 * values.mean()
        ),
        empirical_95_range_coverage=(
            "inside_95_range",
            lambda values:
                100 * values.mean()
        )
    )
    .reset_index()
    .round(2)
)


print("Guidance tier sizing on the locked population")
display(tier_summary)

print("Tier reconciliation")
display(tier_reconciliation)

print("Validation by predicted-price band")
display(predicted_price_band_validation)

# ---------------------------------------------------------
# Final routing rule:
# any predicted benchmark of £50,000+ requires manual review
# ---------------------------------------------------------

tier_population["guidance_tier_final"] = (
    tier_population["guidance_tier"]
)

predicted_50k_plus = (
    tier_population["expected_listed_price_gbp"]
    >= 50000
)

tier_population.loc[
    predicted_50k_plus,
    "guidance_tier_final"
] = "Tier 3 — Manual appraisal"


# ---------------------------------------------------------
# Final tier summary
# ---------------------------------------------------------

final_tier_summary = (
    tier_population
    .groupby("guidance_tier_final")
    .agg(
        listings=(
            "listed_price_gbp",
            "size"
        ),
        aggregate_listed_price_value=(
            "listed_price_gbp",
            "sum"
        ),
        median_listed_price=(
            "listed_price_gbp",
            "median"
        ),
        median_expected_price=(
            "expected_listed_price_gbp",
            "median"
        ),
        predicted_50k_plus_records=(
            "expected_listed_price_gbp",
            lambda values:
                (values >= 50000).sum()
        )
    )
    .reset_index()
)

final_tier_summary["listing_share_percentage"] = (
    100
    * final_tier_summary["listings"]
    / len(tier_population)
)

final_tier_summary[
    "aggregate_listed_price_share_percentage"
] = (
    100
    * final_tier_summary[
        "aggregate_listed_price_value"
    ]
    / tier_population[
        "listed_price_gbp"
    ].sum()
)

final_tier_summary = final_tier_summary[
    [
        "guidance_tier_final",
        "listings",
        "listing_share_percentage",
        "aggregate_listed_price_value",
        "aggregate_listed_price_share_percentage",
        "median_listed_price",
        "median_expected_price",
        "predicted_50k_plus_records"
    ]
].round(2)


final_tier_reconciliation = pd.DataFrame({
    "analytical_population": [
        len(tier_population)
    ],
    "final_tiered_records": [
        final_tier_summary["listings"].sum()
    ],
    "difference": [
        len(tier_population)
        - final_tier_summary["listings"].sum()
    ],
    "all_predicted_50k_plus_in_tier_3": [
        (
            tier_population.loc[
                predicted_50k_plus,
                "guidance_tier_final"
            ]
            == "Tier 3 — Manual appraisal"
        ).all()
    ]
})


print("Final guidance tier sizing")
display(final_tier_summary)

print("Final tier reconciliation")
display(final_tier_reconciliation)

import numpy as np
import pandas as pd

# ---------------------------------------------------------
# Fully supported reference:
# approximate age 1 and 10,000 miles
# ---------------------------------------------------------

reference_row = joint_age_mileage_schedule[
    (
        joint_age_mileage_schedule[
            "approximate_vehicle_age"
        ] == 1
    )
    &
    (
        joint_age_mileage_schedule[
            "mileage_reference_miles"
        ] == 10000
    )
]

assert len(reference_row) == 1

reference_index = float(
    reference_row[
        "joint_adjusted_price_index"
    ].iloc[0]
)

reference_support = int(
    reference_row[
        "training_fingerprint_support"
    ].iloc[0]
)

assert reference_support >= 100


# ---------------------------------------------------------
# Re-anchor every joint index
# ---------------------------------------------------------

joint_schedule_reanchored = (
    joint_age_mileage_schedule.copy()
)

joint_schedule_reanchored[
    "supported_reference_price_index"
] = (
    100
    * joint_schedule_reanchored[
        "joint_adjusted_price_index"
    ]
    / reference_index
)

joint_schedule_reanchored[
    "supported_reference_price_index"
] = (
    joint_schedule_reanchored[
        "supported_reference_price_index"
    ].round(1)
)


# ---------------------------------------------------------
# Presentation matrix:
# hide combinations with fewer than 25 fingerprints
# ---------------------------------------------------------

joint_schedule_reanchored[
    "presentation_index"
] = np.where(
    joint_schedule_reanchored[
        "training_fingerprint_support"
    ] >= 25,
    joint_schedule_reanchored[
        "supported_reference_price_index"
    ],
    np.nan
)

supported_joint_matrix = (
    joint_schedule_reanchored
    .pivot(
        index="approximate_vehicle_age",
        columns="mileage_reference_miles",
        values="presentation_index"
    )
)

supported_joint_matrix.columns = [
    f"{int(mileage):,} miles"
    for mileage in supported_joint_matrix.columns
]


# ---------------------------------------------------------
# Supported headline examples for Step 5.2
# ---------------------------------------------------------

headline_combinations = [
    (1, 10000),
    (3, 30000),
    (5, 60000),
    (7, 60000),
    (10, 100000)
]

headline_joint_schedule = (
    joint_schedule_reanchored[
        joint_schedule_reanchored.apply(
            lambda row: (
                row["approximate_vehicle_age"],
                row["mileage_reference_miles"]
            ) in headline_combinations,
            axis=1
        )
    ]
    [
        [
            "approximate_vehicle_age",
            "model_year",
            "mileage_reference_miles",
            "supported_reference_price_index",
            "training_fingerprint_support",
            "support_status"
        ]
    ]
    .sort_values(
        [
            "approximate_vehicle_age",
            "mileage_reference_miles"
        ]
    )
)


print(
    "Reference: approximate age 1 and "
    "10,000 miles = 100"
)
print(
    "Reference fingerprint support:",
    reference_support
)

print("Supported joint price-index matrix")
display(supported_joint_matrix)

print("Headline supported examples")
display(headline_joint_schedule)

